In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import gc

<h2>Early Concatting, Transforming, and Merging</h2>

<h3>Concat Raw Datasets</h3>

In [36]:
# Concat all facility csv files into a single df

facility_folder = Path('Facility')

df_list = []
for file in facility_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

campd_facility = pd.concat(df_list, ignore_index=True)

output_path = Path('Facility/Concatted File/facility_2019_2024.csv')
campd_facility.to_csv(output_path, index=False)

In [ ]:
# Concat all ny csv files into a single df

ny_folder = Path('NY/Hourly')

df_list = []
for file in ny_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

campd_ny = pd.concat(df_list, ignore_index=True)

# save full operating and non-operating dfs separately

filtered_out_rows_campd_ny = campd_ny[campd_ny["SO2 Mass (lbs)"].isna() & campd_ny["CO2 Mass (short tons)"].isna() & campd_ny["NOx Mass (lbs)"].isna()]
filtered_out_rows_campd_ny.to_csv('NY/Hourly/Concatted File/ny_2019_2024_NonOperating.csv', index=False)

campd_ny = campd_ny[~(campd_ny["SO2 Mass (lbs)"].isna()) & ~(campd_ny["CO2 Mass (short tons)"].isna()) & ~(campd_ny["NOx Mass (lbs)"].isna())]
campd_ny.to_csv('NY/Hourly/Concatted File/ny_2019_2024_Operating.csv', index=False)

del filtered_out_rows_campd_ny, campd_ny
gc.collect()

In [ ]:
# Concat all wy csv files into a single df

wy_folder = Path('WY/Hourly')

df_list = []
for file in wy_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

campd_wy = pd.concat(df_list, ignore_index=True)

# save full operating and non-operating dfs separately

filtered_out_rows_campd_wy = campd_wy[campd_wy["SO2 Mass (lbs)"].isna() & campd_wy["CO2 Mass (short tons)"].isna() & campd_wy["NOx Mass (lbs)"].isna()]
filtered_out_rows_campd_wy.to_csv('WY/Hourly/Concatted File/wy_2019_2024_NonOperating.csv', index=False)

campd_wy = campd_wy[~(campd_wy["SO2 Mass (lbs)"].isna()) & ~(campd_wy["CO2 Mass (short tons)"].isna()) & ~(campd_wy["NOx Mass (lbs)"].isna())]
campd_wy.to_csv('WY/Hourly/Concatted File/wy_2019_2024_Operating.csv', index=False)

del filtered_out_rows_campd_wy, campd_wy
gc.collect()

In [ ]:
# Concat all tx csv files into a single df

tx_folder = Path('TX/Hourly')

df_list = []
for file in tx_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

campd_tx = pd.concat(df_list, ignore_index=True)

# save full operating and non-operating dfs separately

filtered_out_rows_campd_tx = campd_tx[campd_tx["SO2 Mass (lbs)"].isna() & campd_tx["CO2 Mass (short tons)"].isna() & campd_tx["NOx Mass (lbs)"].isna()]
filtered_out_rows_campd_tx.to_csv('TX/Hourly/Concatted File/tx_2019_2024_NonOperating.csv', index=False)

campd_tx = campd_tx[~(campd_tx["SO2 Mass (lbs)"].isna()) & ~(campd_tx["CO2 Mass (short tons)"].isna()) & ~(campd_tx["NOx Mass (lbs)"].isna())]
campd_tx.to_csv('TX/Hourly/Concatted File/tx_2019_2024_Operating.csv', index=False)

del filtered_out_rows_campd_tx, campd_tx
gc.collect()

In [ ]:
# get all EIA data

eia_folder = Path('EIA')

df_list = []
for file in eia_folder.iterdir():
    if file.is_file() and file.name.endswith('.xlsx'):
        # extract the year from filename and add as column
        match = re.search(r'_Y(\d{4})\.xlsx', str(file))
        if match:
            year = int(match.group(1))
            df = pd.read_excel(file, sheet_name="Operable", header=1)
            df["Year"] = year
            df_list.append(df)

eia_generator = pd.concat(df_list, ignore_index=True)

# filter for rows in campd_ny, campd_wy, and campd_tx
eia_generator = eia_generator[eia_generator["Plant Code"].isin(campd_ny["Facility ID"]) | 
                                eia_generator["Plant Code"].isin(campd_wy["Facility ID"]) | 
                                eia_generator["Plant Code"].isin(campd_tx["Facility ID"])]

output_path = Path('EIA/Concatted File/eia_generator_2019_2023.csv')
eia_generator.to_csv(output_path, index=False)

<h3>Transform Emission Data</h3>

In [3]:
# define data type dictionary for campd dataframes
campd_type_dict = {
    "State": "string",
    "Facility Name": "string",
    "Facility ID": "Int64",
    "Unit ID": "string",
    "Associated Stacks": "string",
    "Gross Load (MW)": "float64",
    "Steam Load (1000 lb/hr)": "float64",
    "SO2 Mass (lbs)": "float64",
    "SO2 Mass Measure Indicator": "string",
    "SO2 Rate (lbs/mmBtu)": "float64",
    "SO2 Rate Measure Indicator": "string",
    "CO2 Mass (short tons)": "float64",
    "CO2 Mass Measure Indicator": "string",
    "CO2 Rate (short tons/mmBtu)": "float64",
    "CO2 Rate Measure Indicator": "string",
    "NOx Mass (lbs)": "float64",
    "NOx Mass Measure Indicator": "string",
    "NOx Rate (lbs/mmBtu)": "float64",
    "NOx Rate Measure Indicator": "string",
    "Heat Input (mmBtu)": "float64",
    "Heat Input Measure Indicator": "string",
    "Primary Fuel Type": "string",
    "Secondary Fuel Type": "string",
    "Unit Type": "string",
    "SO2 Controls": "string",
    "NOx Controls": "string",
    "PM Controls": "string",
    "Hg Controls": "string",
    "Program Code": "string",
    "Hour": "Int64",
    "Operating Time": "float64"
}

<h4>Wyoming</h4>

In [7]:
campd_wy = pd.read_csv('WY/Hourly/Concatted File/wy_2019_2024_NonOperating.csv', dtype=campd_type_dict, parse_dates=["Date"])

In [8]:
campd_wy.shape

(321772, 32)

In [9]:
# "Impute" numerical cols where data is missing with 0 (we can reasonably assume that this means 0 heat input, emissions, and gross load)
campd_wy[[
    'Gross Load (MW)', 'Steam Load (1000 lb/hr)', 'SO2 Mass (lbs)',
    'SO2 Rate (lbs/mmBtu)', 'CO2 Mass (short tons)',
    'CO2 Rate (short tons/mmBtu)', 'NOx Mass (lbs)',
    'NOx Rate (lbs/mmBtu)', 'Heat Input (mmBtu)', 'Operating Time'
]] = campd_wy[[
    'Gross Load (MW)', 'Steam Load (1000 lb/hr)', 'SO2 Mass (lbs)',
    'SO2 Rate (lbs/mmBtu)', 'CO2 Mass (short tons)',
    'CO2 Rate (short tons/mmBtu)', 'NOx Mass (lbs)',
    'NOx Rate (lbs/mmBtu)', 'Heat Input (mmBtu)', 'Operating Time'
]].fillna(0)

<h4>New York</h4>

In [10]:
campd_ny = pd.read_csv('NY/Hourly/Concatted File/ny_2019_2024_NonOperating.csv', dtype=campd_type_dict, parse_dates=["Date"])

In [11]:
campd_ny.shape

(11627974, 32)

In [12]:
# "Impute" numerical cols where data is missing with 0 (we can reasonably assume that this means 0 heat input, emissions, and gross load)
campd_ny[[
    'Gross Load (MW)', 'Steam Load (1000 lb/hr)', 'SO2 Mass (lbs)',
    'SO2 Rate (lbs/mmBtu)', 'CO2 Mass (short tons)',
    'CO2 Rate (short tons/mmBtu)', 'NOx Mass (lbs)',
    'NOx Rate (lbs/mmBtu)', 'Heat Input (mmBtu)', 'Operating Time'
]] = campd_ny[[
    'Gross Load (MW)', 'Steam Load (1000 lb/hr)', 'SO2 Mass (lbs)',
    'SO2 Rate (lbs/mmBtu)', 'CO2 Mass (short tons)',
    'CO2 Rate (short tons/mmBtu)', 'NOx Mass (lbs)',
    'NOx Rate (lbs/mmBtu)', 'Heat Input (mmBtu)', 'Operating Time'
]].fillna(0)

<h4>Texas</h4>

In [13]:
campd_tx = pd.read_csv('TX/Hourly/Concatted File/tx_2019_2024_NonOperating.csv', dtype=campd_type_dict, parse_dates=["Date"])

In [14]:
campd_tx.shape

(11224101, 32)

In [15]:
# "Impute" numerical cols where data is missing with 0 (we can reasonably assume that this means 0 heat input, emissions, and gross load)
campd_tx[[
    'Gross Load (MW)', 'Steam Load (1000 lb/hr)', 'SO2 Mass (lbs)',
    'SO2 Rate (lbs/mmBtu)', 'CO2 Mass (short tons)',
    'CO2 Rate (short tons/mmBtu)', 'NOx Mass (lbs)',
    'NOx Rate (lbs/mmBtu)', 'Heat Input (mmBtu)', 'Operating Time'
]] = campd_tx[[
    'Gross Load (MW)', 'Steam Load (1000 lb/hr)', 'SO2 Mass (lbs)',
    'SO2 Rate (lbs/mmBtu)', 'CO2 Mass (short tons)',
    'CO2 Rate (short tons/mmBtu)', 'NOx Mass (lbs)',
    'NOx Rate (lbs/mmBtu)', 'Heat Input (mmBtu)', 'Operating Time'
]].fillna(0)

<h3>Transform Facility Data</h3>

In [16]:
# read facility data and assign data types

campd_facility = pd.read_csv('Facility/Concatted File/facility_2014_2024.csv')

# limit to columns we need
campd_facility = campd_facility[["Year", "State", 'Facility ID', 'Unit ID', "EPA Region", 'Source Category', 'Latitude',
       'Longitude', 'SO2 Phase', 'NOx Phase',
       'Commercial Operation Date', 'Operating Status',
       'Max Hourly HI Rate (mmBtu/hr)',
       'Associated Generators & Nameplate Capacity (MWe)']]
campd_facility = campd_facility.astype({
    'Year': 'Int64',
    'Facility ID': 'Int64',
    'Unit ID': 'string',
    "State": "string",
    'EPA Region': 'Int64',
    "Source Category": 'string',
    'Latitude': 'float64',
    'Longitude': 'float64',
    'SO2 Phase': 'string',
    'NOx Phase': 'string',
    'Commercial Operation Date': 'string',
    'Operating Status': 'string',
    'Max Hourly HI Rate (mmBtu/hr)': 'float64',
    'Associated Generators & Nameplate Capacity (MWe)': 'string'
    })

# filter out rows that are not in campd_ny, campd_wy, or campd_tx
campd_facility = campd_facility[
    campd_facility["Facility ID"].isin(campd_ny["Facility ID"]) |
    campd_facility["Facility ID"].isin(campd_wy["Facility ID"]) |
    campd_facility["Facility ID"].isin(campd_tx["Facility ID"])
]

In [ ]:
# function to extract and sum nameplate capacities in facility data
def extract_and_sum_nameplate_capacity(data):
    data = str(data).strip()  # Convert data to string and remove leading/trailing whitespace
    matches = re.findall(r'\((\d+(?:\.\d+)?)\)', data) # find floats or integers in parentheses (can be float like 191.8 or integer like 170)

    if matches:
        total_sum = 0.0
        for match in matches:
            total_sum += float(match)
        #print("Total sum: ", total_sum)
        return total_sum
    else:
        #print("No matches found")
        return None

campd_facility['Nameplate Capacity (MWe)'] = campd_facility['Associated Generators & Nameplate Capacity (MWe)'].apply(extract_and_sum_nameplate_capacity)
campd_facility.to_csv('Facility/Concatted File/facility_2019_2024.csv', index=False)

In [62]:
campd_facility.isna().sum()

Year                                                   0
State                                                  0
Facility ID                                            0
Unit ID                                                0
EPA Region                                             0
Source Category                                        0
Latitude                                               0
Longitude                                              0
SO2 Phase                                           1530
NOx Phase                                           6366
Commercial Operation Date                              0
Operating Status                                      10
Max Hourly HI Rate (mmBtu/hr)                         19
Associated Generators & Nameplate Capacity (MWe)       0
Nameplate Capacity (MWe)                               0
dtype: int64

<h3>Merge Facility and Emission dfs</h3>

(and do early data cleaning based on this merge)

<h4>Wyoming</h4>

In [18]:
# Look for mismatched Unit IDs between campd_facility and campd_wy for Wyoming facilities

# filter and get unique Facility-Unit pairs
fac_units_1 = campd_facility[
    campd_facility["State"] == "WY"
][["Facility ID", "Unit ID"]].drop_duplicates()

fac_units_2 = campd_wy[["Facility ID", "Unit ID"]].drop_duplicates()

# outer merge with indicator
merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

# filter mismatches
mismatches = merged[merged["_merge"] != "both"]

# count mismatched units per facility
mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

# include the mismatching unit IDs as list
mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

# merge count and unit list
result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)


,Facility ID,Mismatched Unit Count,Mismatching Unit IDs
0,7504,1,[1]
1,55479,1,[1]
2,56319,2,"[001, 1]"
3,56596,1,[1]


In [19]:
# replace all unit IDs that are "001" or "01" with "1" to eliminate mismatches (see above)
campd_wy["Unit ID"] = campd_wy["Unit ID"].replace("001", "1")
campd_wy["Unit ID"] = campd_wy["Unit ID"].replace("01", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("001", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("01", "1")

In [20]:
# check after transformation

fac_units_1 = campd_facility[
    campd_facility["State"] == "WY"
][["Facility ID", "Unit ID"]].drop_duplicates()
fac_units_2 = campd_wy[["Facility ID", "Unit ID"]].drop_duplicates()

merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

mismatches = merged[merged["_merge"] != "both"]

mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)


,Facility ID,Mismatched Unit Count,Mismatching Unit IDs


In [21]:
campd_facility_wy = campd_facility[campd_facility["State"] == "WY"]
campd_facility_wy = campd_facility_wy[["Year", 'Facility ID', 'Unit ID', "EPA Region", 'Source Category', 'Latitude',
       'Longitude', 'SO2 Phase', 'NOx Phase',
       'Commercial Operation Date', 'Operating Status',
       'Max Hourly HI Rate (mmBtu/hr)',
       'Associated Generators & Nameplate Capacity (MWe)', 'Nameplate Capacity (MWe)']]
# add the year column to campd_wy for the join
campd_wy['Year'] = campd_wy['Date'].dt.year
campd_wy_facilities = pd.merge(campd_wy, campd_facility_wy, on=['Facility ID', 'Unit ID', 'Year'], how='left')

In [22]:
campd_wy_facilities.to_csv('WY/Hourly/Concatted File/wy_2019_2024_with_facility_NonOperating.csv', index=False)

<h4>New York</h4>

In [23]:
# Look for mismatched Unit IDs between campd_facility and campd_ny

# filter and get unique Facility-Unit pairs
fac_units_1 = campd_facility[
    campd_facility["State"] == "NY"
][["Facility ID", "Unit ID"]].drop_duplicates()

fac_units_2 = campd_ny[["Facility ID", "Unit ID"]].drop_duplicates()

# outer merge with indicator
merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

# filter mismatches
mismatches = merged[merged["_merge"] != "both"]

# count mismatched units per facility
mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

# include the mismatching unit IDs as list
mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

# merge count and unit list
result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)


,Facility ID,Mismatched Unit Count,Mismatching Unit IDs
0,2494,32,"[CT01-1, CT01-2, CT01-3, CT01-4, CT01-5, CT01-..."
1,2496,3,"[CT0003, CT0004, CT0005]"
2,2499,16,"[CT01-1, CT01-2, CT01-3, CT01-4, CT01-5, CT01-..."
3,2503,6,"[BLR114, BLR115, BLR116, BLR117, BLR118, CT0001]"
4,2512,1,[UGT001]
5,7314,2,"[001, 1]"
6,10025,1,[4B]
7,10619,1,[1]
8,10803,2,"[1, 2]"
9,50368,2,"[CT1, CT2]"


In [24]:
# replace all unit IDs that are "001" or "01" with "1" to eliminate mismatches (see above)
campd_ny["Unit ID"] = campd_ny["Unit ID"].replace("001", "1")
campd_ny["Unit ID"] = campd_ny["Unit ID"].replace("01", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("001", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("01", "1")

In [25]:
campd_facility_ny = campd_facility[campd_facility["State"] == "NY"]
display(campd_facility_ny[campd_facility_ny["Facility ID"] == 2500].drop_duplicates(subset=["Unit ID"])[["Facility ID","Unit ID"]])
campd_ny['Year'] = campd_ny['Date'].dt.year
display(campd_ny[campd_ny["Facility ID"] == 2500].drop_duplicates(subset=["Unit ID"])[["Facility ID","Unit ID"]])

,Facility ID,Unit ID
14,2500,10
15,2500,20
16,2500,30
17,2500,CT0001
18,2500,CT0004
19,2500,CT0005
20,2500,CT0006
21,2500,CT0007
22,2500,CT0008
23,2500,CT0009


,Facility ID,Unit ID
279759,2500,10
286329,2500,20
291692,2500,30
299798,2500,CT0001
304181,2500,CT0004
308573,2500,CT0005
312965,2500,CT0006
317357,2500,CT0007
321749,2500,CT0008
326141,2500,CT0009


In [26]:
# check after transformation

# filter and get unique Facility-Unit pairs
fac_units_1 = campd_facility[
    campd_facility["State"] == "NY"
][["Facility ID", "Unit ID"]].drop_duplicates()

fac_units_2 = campd_ny[["Facility ID", "Unit ID"]].drop_duplicates()

# outer merge with indicator
merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

# filter mismatches
mismatches = merged[merged["_merge"] != "both"]

# count mismatched units per facility
mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

# include the mismatching unit IDs as list
mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

# merge count and unit list
result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)

# the 001 -> 1 transformation fixed only one unit mismatch


,Facility ID,Mismatched Unit Count,Mismatching Unit IDs
0,2494,32,"[CT01-1, CT01-2, CT01-3, CT01-4, CT01-5, CT01-..."
1,2496,3,"[CT0003, CT0004, CT0005]"
2,2499,16,"[CT01-1, CT01-2, CT01-3, CT01-4, CT01-5, CT01-..."
3,2503,6,"[BLR114, BLR115, BLR116, BLR117, BLR118, CT0001]"
4,2512,1,[UGT001]
5,10025,1,[4B]
6,10619,1,[1]
7,10803,2,"[1, 2]"
8,50368,2,"[CT1, CT2]"
9,50472,3,"[BLR05, R1B01, R1B02]"


In [27]:
campd_facility_ny = campd_facility[campd_facility["State"] == "NY"]
campd_facility_ny = campd_facility_ny[["Year", 'Facility ID', 'Unit ID', "EPA Region", 'Source Category', 'Latitude',
       'Longitude', 'SO2 Phase', 'NOx Phase',
       'Commercial Operation Date', 'Operating Status',
       'Max Hourly HI Rate (mmBtu/hr)',
       'Associated Generators & Nameplate Capacity (MWe)', 'Nameplate Capacity (MWe)']]
# add the year column to campd_ny for the join
campd_ny['Year'] = campd_ny['Date'].dt.year
campd_ny_facilities = pd.merge(campd_ny, campd_facility_ny, on=['Facility ID', 'Unit ID', 'Year'], how='left')

In [29]:
campd_ny_facilities.to_csv('NY/Hourly/Concatted File/ny_2019_2024_with_facility_NonOperating.csv', index=False)

<h4>Texas</h4>

In [30]:
# Look for mismatched Unit IDs between campd_facility and campd_tx

# filter and get unique Facility-Unit pairs
fac_units_1 = campd_facility[
    campd_facility["State"] == "TX"
][["Facility ID", "Unit ID"]].drop_duplicates()

fac_units_2 = campd_tx[["Facility ID", "Unit ID"]].drop_duplicates()

# outer merge with indicator
merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

# filter mismatches
mismatches = merged[merged["_merge"] != "both"]

# count mismatched units per facility
mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

# include the mismatching unit IDs as list
mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

# merge count and unit list
result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)


,Facility ID,Mismatched Unit Count,Mismatching Unit IDs
0,9,1,[CTG-1]
1,3459,1,[2]
2,3464,1,[GBY5]
3,3468,2,"[SRB1, SRB2]"
4,3470,1,[CTSC]
5,3476,1,[4]
6,3492,6,"[CT1, CT2, CT3, CT4, CT5, CT6]"
7,3494,2,"[5, 6]"
8,6139,1,[2]
9,8063,4,"[CT1, CT2, CT3, CT4]"


In [31]:
# replace all unit IDs that are "001" or "01" with "1" to eliminate mismatches (see above)
campd_tx["Unit ID"] = campd_tx["Unit ID"].replace("001", "1")
campd_tx["Unit ID"] = campd_tx["Unit ID"].replace("01", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("001", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("01", "1")

In [32]:
# check after transformation

# filter and get unique Facility-Unit pairs
fac_units_1 = campd_facility[
    campd_facility["State"] == "TX"
][["Facility ID", "Unit ID"]].drop_duplicates()

fac_units_2 = campd_tx[["Facility ID", "Unit ID"]].drop_duplicates()

# outer merge with indicator
merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

# filter mismatches
mismatches = merged[merged["_merge"] != "both"]

# count mismatched units per facility
mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

# include the mismatching unit IDs as list
mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

# merge count and unit list
result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)

# the 001 -> 1 transformation fixed none here

,Facility ID,Mismatched Unit Count,Mismatching Unit IDs
0,9,1,[CTG-1]
1,3459,1,[2]
2,3464,1,[GBY5]
3,3468,2,"[SRB1, SRB2]"
4,3470,1,[CTSC]
5,3476,1,[4]
6,3492,6,"[CT1, CT2, CT3, CT4, CT5, CT6]"
7,3494,2,"[5, 6]"
8,6139,1,[2]
9,8063,4,"[CT1, CT2, CT3, CT4]"


In [33]:
campd_facility_tx = campd_facility[campd_facility["State"] == "TX"]
campd_facility_tx = campd_facility_tx[["Year", 'Facility ID', 'Unit ID', "EPA Region", 'Source Category', 'Latitude',
       'Longitude', 'SO2 Phase', 'NOx Phase',
       'Commercial Operation Date', 'Operating Status',
       'Max Hourly HI Rate (mmBtu/hr)',
       'Associated Generators & Nameplate Capacity (MWe)', 'Nameplate Capacity (MWe)']]
# add the year column to campd_ny for the join
campd_tx['Year'] = campd_tx['Date'].dt.year
campd_tx_facilities = pd.merge(campd_tx, campd_facility_tx, on=['Facility ID', 'Unit ID', 'Year'], how='left')

In [36]:
campd_tx_facilities.to_csv('TX/Hourly/Concatted File/tx_2019_2024_with_facility_NonOperating.csv', index=False)

<h3>Transform EIA-860 Data</h3>

(has additional data about generator nameplate capacities and age, but needs to be aggregated to a unit level)

In [71]:
# read eia generator data
eia_generator = pd.read_csv('EIA/Concatted File/eia_generator_2019_2023.csv')

# these cols have blank strings (" ") that should be nan
columns_to_clean = [
    "Nameplate Power Factor", "Summer Capacity (MW)", "Winter Capacity (MW)",
    "Minimum Load (MW)", "Planned Retirement Year"
]

# replace blank strings with nan (any strings with only whitespace)
eia_generator[columns_to_clean] = eia_generator[columns_to_clean].replace(r'^\s*$', np.nan, regex=True)

# now define full dtype dictionary
eia_generator_type_dict = {
    "Year": "Int64", 
    "Utility ID": "string",
    "Utility Name": "string",
    "Plant Code": "Int64",  
    "Plant Name": "string",
    "State": "string",
    "Generator ID": "string",
    "Technology": "string",
    "Prime Mover": "string",
    "Nameplate Capacity (MW)": "float64",
    "Nameplate Power Factor": "float64", 
    "Summer Capacity (MW)": "float64",  
    "Winter Capacity (MW)": "float64",
    "Minimum Load (MW)": "float64",
    "Operating Year": "Int64",
    "Planned Retirement Year": "float64"
}

# filter needed cols
eia_generator = eia_generator[["Year","Utility ID","Utility Name", "Plant Code", "Plant Name", "State", "Generator ID", "Technology", "Prime Mover", "Nameplate Capacity (MW)", "Nameplate Power Factor", "Summer Capacity (MW)", "Winter Capacity (MW)", "Minimum Load (MW)", "Operating Year", "Planned Retirement Year"]]

# cast all dtypes according to the dictionary
eia_generator = eia_generator.astype(eia_generator_type_dict)


<h4>old manual version without crosswalk</h4>

In [75]:
# Transform and aggregate to a unit-level (using facility data to add unit IDs)

# extract Generator IDs from the associated string (in campd_facility) and put them in a list 
def extract_generator_ids(data):
    data = str(data).strip()
    # format should be e. g. "G1 (170), G2 (191.8)"
    matches = re.findall(r'([A-Za-z0-9]+)\s*\(\d+(?:\.\d+)?\)', data)
    return matches if matches else []

campd_facility["Generator ID List"] = campd_facility["Associated Generators & Nameplate Capacity (MWe)"].apply(extract_generator_ids)

# explode so each row is Unit ID + one Generator ID -> create a year, facility, unit, generator mapping (a unit may have multiple generators)
unit_generator_map = campd_facility[["Year", "Facility ID", "Unit ID", "Generator ID List",]].explode("Generator ID List")
unit_generator_map = unit_generator_map.rename(columns={"Generator ID List": "Generator ID"})
display(unit_generator_map)

# merge with eia generator-level dataset
# (first rename Plant Code in eia_generator to Facility ID to match unit_generator_map)
eia_generator_renamed = eia_generator.rename(columns={"Plant Code": "Facility ID"})
# merge on Facility ID and Generator ID
joined = unit_generator_map.merge(eia_generator_renamed, on=["Year","Facility ID", "Generator ID"], how="inner")
display(joined)

,Year,Facility ID,Unit ID,Generator ID
0,2014,127,1,1
1,2014,298,LIM1,1
2,2014,298,LIM2,2
3,2014,2480,1,1
4,2014,2480,2,2
...,...,...,...,...
6956,2024,66604,CT-3,3
6957,2024,66604,CT-4,4
6958,2024,66604,CT-5,5
6959,2024,66604,CT-6,6


,Year,Facility ID,Unit ID,Generator ID,Utility ID,Utility Name,Plant Name,State,Technology,Prime Mover,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year,Planned Retirement Year
0,2019,127,1,1,15474,Public Service Co of Oklahoma,Oklaunion,TX,Conventional Steam Coal,ST,720.0,0.90,650.0,650.0,215.0,1986,2020.0
1,2019,298,LIM1,1,54888,NRG Texas Power LLC,Limestone,TX,Conventional Steam Coal,ST,893.0,0.94,831.0,831.0,300.0,1985,2030.0
2,2019,298,LIM2,2,54888,NRG Texas Power LLC,Limestone,TX,Conventional Steam Coal,ST,956.8,0.92,858.0,858.0,300.0,1986,2030.0
3,2019,2480,1,1,58971,Danskammer Energy,Danskammer Generating Station,NY,Natural Gas Steam Turbine,ST,72.0,0.85,70.7,68.2,20.0,1951,NaN
4,2019,2480,2,2,58971,Danskammer Energy,Danskammer Generating Station,NY,Natural Gas Steam Turbine,ST,73.5,0.85,65.7,65.5,20.0,1954,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3414,2023,60460,CT2,CH2,61199,"Peaker Power, LLC",Chamon Power LLC,TX,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,43.0,46.0,20.0,2017,NaN
3415,2023,60925,CT1,1A,55937,Entergy Texas Inc.,Montgomery County,TX,Natural Gas Fired Combined Cycle,CT,250.0,0.85,250.0,250.0,NaN,2021,NaN
3416,2023,60925,CT1,1C,55937,Entergy Texas Inc.,Montgomery County,TX,Natural Gas Fired Combined Cycle,CA,500.0,0.85,500.0,500.0,NaN,2021,NaN
3417,2023,60925,CT2,1C,55937,Entergy Texas Inc.,Montgomery County,TX,Natural Gas Fired Combined Cycle,CA,500.0,0.85,500.0,500.0,NaN,2021,NaN


In [76]:
# aggregate the resulting generator-level df to a unit-level df

agg_funcs = {
    'Nameplate Capacity (MW)': 'sum',
    'Summer Capacity (MW)': 'sum',
    'Winter Capacity (MW)': 'sum',
    'Minimum Load (MW)': 'sum',
    'Operating Year': ['min', 'max', 'mean'], # get operating year of the oldest (min) and newest (max) generator in the unit as well as the average across all gens
    'Planned Retirement Year': 'max',
}

unit_level = joined.groupby(['Year', 'Facility ID', 'Unit ID']).agg(agg_funcs).reset_index()

display(unit_level)

# flatten multi-index columns (only keep the aggregation operator (min, max, mean) for Operating Year)
unit_level.columns = [
    f"{col[0]}_{col[1]}" if col[0] == "Operating Year" else col[0]
    for col in unit_level.columns.to_flat_index()
]

unit_level

Year Facility ID Unit ID Nameplate Capacity (MW) Summer Capacity (MW)  \
                                                   sum                  sum   
0     2019         127       1                   720.0                650.0   
1     2019         298    LIM1                   893.0                831.0   
2     2019         298    LIM2                   956.8                858.0   
3     2019        2480       1                    72.0                 70.7   
4     2019        2480       2                    73.5                 65.7   
...    ...         ...     ...                     ...                  ...   
2486  2023       60459     CT2                    60.5                 43.0   
2487  2023       60460     CT1                    60.5                 43.0   
2488  2023       60460     CT2                    60.5                 43.0   
2489  2023       60925     CT1                   750.0                750.0   
2490  2023       60925     CT2                   750.0                750.0   

     Winter Capacity (MW) Minimum Load (MW) Operating Year                \
                      sum               sum            min   max    mean   
0                   650.0             215.0           1986  1986  1986.0   
1                   831.0             300.0           1985  1985  1985.0   
2                   858.0             300.0           1986  1986  1986.0   
3                    68.2              20.0           1951  1951  1951.0   
4                    65.5              20.0           1954  1954  1954.0   
...                   ...               ...            ...   ...     ...   
2486                 46.0              20.0           2017  2017  2017.0   
2487                 46.0              20.0           2017  2017  2017.0   
2488                 46.0              20.0           2017  2017  2017.0   
2489                750.0               0.0           2021  2021  2021.0   
2490                750.0               0.0           2021  2021  2021.0   

     Planned Retirement Year  
                         max  
0                     2020.0  
1                     2030.0  
2                     2030.0  
3                        NaN  
4                        NaN  
...                      ...  
2486                     NaN  
2487                     NaN  
2488                     NaN  
2489                     NaN  
2490                     NaN  

[2491 rows x 11 columns]

,Year,Facility ID,Unit ID,Nameplate Capacity (MW),Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year_min,Operating Year_max,Operating Year_mean,Planned Retirement Year
0,2019,127,1,720.0,650.0,650.0,215.0,1986,1986,1986.0,2020.0
1,2019,298,LIM1,893.0,831.0,831.0,300.0,1985,1985,1985.0,2030.0
2,2019,298,LIM2,956.8,858.0,858.0,300.0,1986,1986,1986.0,2030.0
3,2019,2480,1,72.0,70.7,68.2,20.0,1951,1951,1951.0,NaN
4,2019,2480,2,73.5,65.7,65.5,20.0,1954,1954,1954.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2486,2023,60459,CT2,60.5,43.0,46.0,20.0,2017,2017,2017.0,NaN
2487,2023,60460,CT1,60.5,43.0,46.0,20.0,2017,2017,2017.0,NaN
2488,2023,60460,CT2,60.5,43.0,46.0,20.0,2017,2017,2017.0,NaN
2489,2023,60925,CT1,750.0,750.0,750.0,0.0,2021,2021,2021.0,NaN


In [77]:
# set data types again for the final unit_level dataframe
unit_level = unit_level.astype({
    'Year': 'Int64',
    'Facility ID': 'Int64',
    'Unit ID': 'string',
    'Nameplate Capacity (MW)': 'float64',
    'Summer Capacity (MW)': 'float64',
    'Winter Capacity (MW)': 'float64',
    'Minimum Load (MW)': 'float64',
    'Operating Year_min': 'Int64',
    'Operating Year_max': 'Int64',
    'Operating Year_mean': 'Int64',
    'Planned Retirement Year': 'Int64'
})

In [78]:
# Artificially extend the unit_level dataframe to include 2024 data (just copy the 2023 data and set Year to 2024)
unit_level_2023 = unit_level[unit_level["Year"] == 2023].copy()
unit_level_2024 = unit_level_2023.copy()
unit_level_2024["Year"] = 2024
unit_level_extended = pd.concat([unit_level, unit_level_2024], ignore_index=True)

In [79]:
# Calc min, max, and mean ages of the generators in the unit
unit_level_extended['unit_age_min'] = unit_level_extended["Year"] - unit_level_extended['Operating Year_min']
unit_level_extended['unit_age_max'] = unit_level_extended["Year"] - unit_level_extended['Operating Year_max']
unit_level_extended['unit_age_mean'] = unit_level_extended["Year"] - unit_level_extended['Operating Year_mean']

In [71]:
# save to csv
unit_level_extended.to_csv('EIA/Concatted File/eia_unit_level_2019_2024.csv', index=False)

In [6]:
unit_level_extended_manually = pd.read_csv('EIA/Concatted File/eia_unit_level_2019_2024.csv')
unit_level_extended_manually

,Year,Facility ID,Unit ID,Nameplate Capacity (MW),Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year_min,Operating Year_max,Operating Year_mean,Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean
0,2019,127,1,720.0,650.0,650.0,215.0,1986,1986,1986,2020.0,33,33,33
1,2019,298,LIM1,893.0,831.0,831.0,300.0,1985,1985,1985,2030.0,34,34,34
2,2019,298,LIM2,956.8,858.0,858.0,300.0,1986,1986,1986,2030.0,33,33,33
3,2019,2480,1,72.0,70.7,68.2,20.0,1951,1951,1951,NaN,68,68,68
4,2019,2480,2,73.5,65.7,65.5,20.0,1954,1954,1954,NaN,65,65,65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2974,2024,60459,CT2,60.5,43.0,46.0,20.0,2017,2017,2017,NaN,7,7,7
2975,2024,60460,CT1,60.5,43.0,46.0,20.0,2017,2017,2017,NaN,7,7,7
2976,2024,60460,CT2,60.5,43.0,46.0,20.0,2017,2017,2017,NaN,7,7,7
2977,2024,60925,CT1,750.0,750.0,750.0,0.0,2021,2021,2021,NaN,3,3,3


<h3>Merge Emission data with EIA data</h3>

In [ ]:
campd_wy_facilities = pd.read_csv('WY/Hourly/Concatted File/wy_2019_2024_with_facility_NonOperating.csv', dtype=campd_type_dict, date_format=["Date"])
campd_ny_facilities = pd.read_csv('NY/Hourly/Concatted File/ny_2019_2024_with_facility_NonOperating.csv', dtype=campd_type_dict, date_format=["Date"])
campd_tx_facilities = pd.read_csv('TX/Hourly/Concatted File/tx_2019_2024_with_facility_NonOperating.csv', dtype=campd_type_dict, date_format=["Date"])

In [7]:
complete_campd_wy = pd.merge(campd_wy_facilities, unit_level_extended_manually, on=['Year', 'Facility ID', 'Unit ID'], how='left')
complete_campd_ny = pd.merge(campd_ny_facilities, unit_level_extended_manually, on=['Year', 'Facility ID', 'Unit ID'], how='left')
complete_campd_tx = pd.merge(campd_tx_facilities, unit_level_extended_manually, on=['Year', 'Facility ID', 'Unit ID'], how='left')

In [8]:
# Filter where the values are different (excluding NaNs where both are the same)
mismatch_df = complete_campd_ny[
    complete_campd_ny["Nameplate Capacity (MWe)"] != complete_campd_ny["Nameplate Capacity (MW)"]
][["Year", "Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"]].drop_duplicates()

# Count where either column is NaN
nan_mwe = complete_campd_ny["Nameplate Capacity (MWe)"].isna().sum()
nan_mw = complete_campd_ny["Nameplate Capacity (MW)"].isna().sum()

# Display results
print("Unique mismatching rows (with year):")
display(mismatch_df)

print(f"\nNaN counts:")
print(f"Nameplate Capacity (MWe): {nan_mwe}")
print(f"Nameplate Capacity (MW): {nan_mw}")

Unique mismatching rows (with year):


,Year,Nameplate Capacity (MWe),Nameplate Capacity (MW)
34832,2019,392.5,360.0
44528,2019,16.3,18.0
51836,2019,156.3,156.2
58904,2019,NaN,NaN
291692,2019,1027.7,1027.0
...,...,...,...
11483462,2024,441.0,407.2
11504950,2024,60.5,NaN
11546673,2024,54.0,NaN
11578607,2024,534.0,535.0



NaN counts:
Nameplate Capacity (MWe): 3216011
Nameplate Capacity (MW): 5095669


In [9]:
# Create a column for absolute difference
complete_campd_ny["Capacity Difference"] = (
    complete_campd_ny["Nameplate Capacity (MWe)"] - complete_campd_ny["Nameplate Capacity (MW)"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_ny.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

# Calculate average difference
average_difference = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference: {average_difference:.2f} MW")

# Sort by largest difference
top_differences = diff_df.sort_values(by="Capacity Difference", ascending=False)

# Display top 10 mismatches
display(top_differences[["Year", "Facility ID", "Unit ID", 
                         "Nameplate Capacity (MWe)", "Nameplate Capacity (MW)", 
                         "Capacity Difference"]].head(10))


Average Capacity Difference: 2.14 MW


,Year,Facility ID,Unit ID,Nameplate Capacity (MWe),Nameplate Capacity (MW),Capacity Difference
5784492,2021,55375,CT2,350.0,425.0,75.0
1797391,2019,55375,CT2,350.0,425.0,75.0
1797378,2019,55375,CT2,350.0,425.0,75.0
1797379,2019,55375,CT2,350.0,425.0,75.0
1797380,2019,55375,CT2,350.0,425.0,75.0
1797381,2019,55375,CT2,350.0,425.0,75.0
1797382,2019,55375,CT2,350.0,425.0,75.0
1797383,2019,55375,CT2,350.0,425.0,75.0
1797384,2019,55375,CT2,350.0,425.0,75.0
1797385,2019,55375,CT2,350.0,425.0,75.0


In [10]:
# Create a column for absolute difference
complete_campd_ny["Capacity Difference"] = (
    complete_campd_ny["Nameplate Capacity (MWe)"] - complete_campd_ny["Nameplate Capacity (MW)"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_ny.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

# Calculate average difference
average_difference_ny = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference NY: {average_difference_ny:.2f} MW")

# Create a column for absolute difference
complete_campd_tx["Capacity Difference"] = (
    complete_campd_tx["Nameplate Capacity (MWe)"] - complete_campd_tx["Nameplate Capacity (MW)"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_tx.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

# Calculate average difference
average_difference_tx = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference TX: {average_difference_tx:.2f} MW")

# Create a column for absolute difference
complete_campd_wy["Capacity Difference"] = (
    complete_campd_wy["Nameplate Capacity (MWe)"] - complete_campd_wy["Nameplate Capacity (MW)"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_wy.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

# Calculate average difference
average_difference_wy = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference WY: {average_difference_wy:.2f} MW")


print("Average Difference across all: ", np.mean([average_difference_ny, average_difference_tx, average_difference_wy]))



Average Capacity Difference NY: 2.14 MW
Average Capacity Difference TX: 8.75 MW
Average Capacity Difference WY: 17.32 MW
Average Difference across all:  9.401167353237314


In [11]:
# Create a column for absolute difference
complete_campd_wy["Capacity Difference"] = (
    complete_campd_wy["Nameplate Capacity (MWe)"] - complete_campd_wy["Nameplate Capacity (MW)"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_wy.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

# Calculate average difference
average_difference = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference: {average_difference:.2f} MW")

# Sort by largest difference
top_differences = diff_df.sort_values(by="Capacity Difference", ascending=False)

# Display top 10 mismatches
display(top_differences[["Year", "Facility ID", "Unit ID", 
                         "Nameplate Capacity (MWe)", "Nameplate Capacity (MW)", 
                         "Capacity Difference"]].drop_duplicates(subset=["Year","Unit ID"]).head(10))


Average Capacity Difference: 17.32 MW


,Year,Facility ID,Unit ID,Nameplate Capacity (MWe),Nameplate Capacity (MW),Capacity Difference
194506,2022,8066,BW72,530.0,617.0,87.0
26916,2019,8066,BW72,530.0,617.0,87.0
138227,2021,8066,BW72,530.0,617.0,87.0
83115,2020,8066,BW72,530.0,617.0,87.0
137973,2021,8066,BW71,530.0,608.3,78.3
138245,2021,8066,BW73,530.0,608.3,78.3
193802,2022,8066,BW71,530.0,608.3,78.3
83542,2020,8066,BW74,530.0,608.3,78.3
138507,2021,8066,BW74,530.0,608.3,78.3
27363,2019,8066,BW73,530.0,608.3,78.3


In [ ]:
# fill EIA columns with na if it capacity difference exceeds a certain threshold

# rename the EIA column to make it clearer
for df in [complete_campd_ny, complete_campd_wy, complete_campd_tx]:
    df.rename(columns={
        "Nameplate Capacity (MW)": "Nameplate Capacity (MW) - EIA"
    }, inplace=True)

threshold = 30 
columns_to_null = [
    "Nameplate Capacity (MW) - EIA", 
    "Summer Capacity (MW)",
    "Winter Capacity (MW)",
    "Minimum Load (MW)"
]

# Recalculate capacity difference
complete_campd_wy["Capacity Difference"] = (
    complete_campd_wy["Nameplate Capacity (MWe)"] - complete_campd_wy["Nameplate Capacity (MW) - EIA"]
).abs()

# Identify rows with unreasonable differences
condition = complete_campd_wy["Capacity Difference"] > threshold

# Set selected columns to NaN where condition is met
complete_campd_wy.loc[condition, columns_to_null] = np.nan

# Recalculate capacity difference
complete_campd_tx["Capacity Difference"] = (
    complete_campd_tx["Nameplate Capacity (MWe)"] - complete_campd_tx["Nameplate Capacity (MW) - EIA"]
).abs()

# Identify rows with unreasonable differences
condition = complete_campd_tx["Capacity Difference"] > threshold

# Set selected columns to NaN where condition is met
complete_campd_tx.loc[condition, columns_to_null] = np.nan

# Recalculate capacity difference
complete_campd_ny["Capacity Difference"] = (
    complete_campd_ny["Nameplate Capacity (MWe)"] - complete_campd_ny["Nameplate Capacity (MW) - EIA"]
).abs()

# Identify rows with unreasonable differences
condition = complete_campd_ny["Capacity Difference"] > threshold

# Set selected columns to NaN where condition is met
complete_campd_ny.loc[condition, columns_to_null] = np.nan

In [13]:
navals = complete_campd_ny[["Nameplate Capacity (MW) - EIA", "Nameplate Capacity (MWe)"]].isna().sum()
navals_percent = (navals / len(complete_campd_ny)) * 100
navals_percent.sort_values(ascending=False)


Nameplate Capacity (MW) - EIA    44.999602
Nameplate Capacity (MWe)         27.657535
dtype: float64

In [14]:
navals = complete_campd_tx[["Nameplate Capacity (MW) - EIA", "Nameplate Capacity (MWe)"]].isna().sum()
navals_percent = (navals / len(complete_campd_tx)) * 100
navals_percent.sort_values(ascending=False)


Nameplate Capacity (MW) - EIA    36.397605
Nameplate Capacity (MWe)          5.011920
dtype: float64

In [15]:
navals = complete_campd_wy[["Nameplate Capacity (MW) - EIA", "Nameplate Capacity (MWe)"]].isna().sum()
navals_percent = (navals / len(complete_campd_wy)) * 100
navals_percent.sort_values(ascending=False)


Nameplate Capacity (MW) - EIA    28.482901
Nameplate Capacity (MWe)          0.000000
dtype: float64

In [16]:
complete_campd_wy.to_csv('WY/Hourly/Concatted File/wy_2019_2024_with_manualEIA_NonOperating.csv', index=False)
complete_campd_ny.to_csv('NY/Hourly/Concatted File/ny_2019_2024_with_manualEIA_NonOperating.csv', index=False)
complete_campd_tx.to_csv('TX/Hourly/Concatted File/tx_2019_2024_with_manualEIA_NonOperating.csv', index=False)

<h2>Define possible features</h2>

In [72]:
relevant_features = [
    "Year", "Month", "Hour", "Gross Load (MW)", "Heat Input (mmBtu)", 
    "Source Category", "NOx Phase", "Unit Type", 
    "Primary Fuel Type", "Secondary Fuel Type", "NOx Controls", "SO2 Controls",
    "County Code", "NOx Mass (lbs)", "Operating Time", "Program Code", "EPA Region",
    "CO2 Mass (short tons)", "SO2 Mass (lbs)", "Nameplate Capacity (MWe)", "Latitude", "Longitude", "Operating Status",   
    "Max Hourly HI Rate (mmBtu/hr)"
]

target_variables = ["NOx Mass (lbs)", "CO2 Mass (short tons)", "SO2 Mass (lbs)"]

<h2>Data Cleaning of Final Datasets</h2>

In [17]:
# relevant columns
complete_campd_columns = [
    "State",
    "Facility Name",
    "Facility ID",
    "Unit ID",
    "Gross Load (MW)",
    "Steam Load (1000 lb/hr)",
    "SO2 Mass (lbs)",
    "SO2 Mass Measure Indicator",
    "SO2 Rate (lbs/mmBtu)",
    "SO2 Rate Measure Indicator",
    "CO2 Mass (short tons)",
    "CO2 Mass Measure Indicator",
    "CO2 Rate (short tons/mmBtu)",
    "CO2 Rate Measure Indicator",
    "NOx Mass (lbs)",
    "NOx Mass Measure Indicator",
    "NOx Rate (lbs/mmBtu)",
    "NOx Rate Measure Indicator",
    "Heat Input (mmBtu)",
    "Heat Input Measure Indicator",
    "Primary Fuel Type",
    "Secondary Fuel Type",
    "Unit Type",
    "SO2 Controls",
    "NOx Controls",
    "Program Code",
    "Hour",
    "Year",
    "Date",
    "Operating Time",
    "Source Category",
    "Latitude",
    "Longitude",
    "SO2 Phase",
    "NOx Phase",
    "Operating Status",
    "Max Hourly HI Rate (mmBtu/hr)",
    "Associated Generators & Nameplate Capacity (MWe)",
    "Nameplate Capacity (MWe)",
    "Nameplate Capacity (MW) - EIA",
    "Summer Capacity (MW)",
    "Winter Capacity (MW)",
    "Minimum Load (MW)",
    "Planned Retirement Year",
    "unit_age_min",
    "unit_age_max",
    "unit_age_mean"
]

# Filtered dtype dictionary
complete_campd_type_dict = {
    "State": "string",
    "Facility Name": "string",
    "Facility ID": "Int64",
    "Unit ID": "string",
    "Gross Load (MW)": "float64",
    "Steam Load (1000 lb/hr)": "float64",
    "SO2 Mass (lbs)": "float64",
    "SO2 Mass Measure Indicator": "string",
    "SO2 Rate (lbs/mmBtu)": "float64",
    "SO2 Rate Measure Indicator": "string",
    "CO2 Mass (short tons)": "float64",
    "CO2 Mass Measure Indicator": "string",
    "CO2 Rate (short tons/mmBtu)": "float64",
    "CO2 Rate Measure Indicator": "string",
    "NOx Mass (lbs)": "float64",
    "NOx Mass Measure Indicator": "string",
    "NOx Rate (lbs/mmBtu)": "float64",
    "NOx Rate Measure Indicator": "string",
    "Heat Input (mmBtu)": "float64",
    "Heat Input Measure Indicator": "string",
    "Primary Fuel Type": "string",
    "Secondary Fuel Type": "string",
    "Unit Type": "string",
    "SO2 Controls": "string",
    "NOx Controls": "string",
    "Program Code": "string",
    "Hour": "Int64",
    "Year": "Int64",
    "Operating Time": "float64",
    "Source Category": "string",
    "Latitude": "float64",
    "Longitude": "float64",
    "SO2 Phase": "string",
    "NOx Phase": "string",
    "Operating Status": "string",
    "Max Hourly HI Rate (mmBtu/hr)": "float64",
    "Associated Generators & Nameplate Capacity (MWe)": "string",
    "Nameplate Capacity (MW)": "float64",
    "Nameplate Capacity (MW) - EIA": "float64",
    "Summer Capacity (MW)": "float64",
    "Winter Capacity (MW)": "float64",
    "Minimum Load (MW)": "float64",
    "Planned Retirement Year": "Int64",
    "unit_age_min": "Int64",
    "unit_age_max": "Int64",
    "unit_age_mean": "float64"
}


<h3>Wyoming</h3>

In [19]:
complete_campd_wy = pd.read_csv(
    "WY/Hourly/Concatted File/wy_2019_2024_with_manualEIA_NonOperating.csv",  
    usecols=complete_campd_columns,
    dtype=complete_campd_type_dict,
    parse_dates=["Date"]
)

# extract month and day as new col from date
complete_campd_wy["Month"] = complete_campd_wy["Date"].dt.month
complete_campd_wy["Day"] = complete_campd_wy["Date"].dt.month

display(complete_campd_wy.head())
print(complete_campd_wy.dtypes)

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Nameplate Capacity (MW) - EIA,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day
0,WY,Dave Johnston,4158,BW41,2019-02-18,11,0.0,0.0,0.0,0.0,...,133.6,93.0,99.0,45.0,<NA>,60,60,60.0,2,2
1,WY,Dave Johnston,4158,BW41,2019-02-18,12,0.0,0.0,0.0,0.0,...,133.6,93.0,99.0,45.0,<NA>,60,60,60.0,2,2
2,WY,Dave Johnston,4158,BW41,2019-02-18,13,0.0,0.0,0.0,0.0,...,133.6,93.0,99.0,45.0,<NA>,60,60,60.0,2,2
3,WY,Dave Johnston,4158,BW41,2019-02-18,14,0.0,0.0,0.0,0.0,...,133.6,93.0,99.0,45.0,<NA>,60,60,60.0,2,2
4,WY,Dave Johnston,4158,BW41,2019-02-18,15,0.0,0.0,0.0,0.0,...,133.6,93.0,99.0,45.0,<NA>,60,60,60.0,2,2


State                                               string[python]
Facility Name                                       string[python]
Facility ID                                                  Int64
Unit ID                                             string[python]
Date                                                datetime64[ns]
Hour                                                         Int64
Operating Time                                             float64
Gross Load (MW)                                            float64
Steam Load (1000 lb/hr)                                    float64
SO2 Mass (lbs)                                             float64
SO2 Mass Measure Indicator                          string[python]
SO2 Rate (lbs/mmBtu)                                       float64
SO2 Rate Measure Indicator                          string[python]
CO2 Mass (short tons)                                      float64
CO2 Mass Measure Indicator                          string[pyt

In [20]:
print(complete_campd_wy.shape)
null_wy = complete_campd_wy.isnull().sum()
null_wy = null_wy[null_wy > 0].sort_values(ascending=False)
display(null_wy)

# Give null vals as percentage of total rows
total_rows = complete_campd_wy.shape[0]
null_percentages = (null_wy / total_rows) * 100
null_percentages.sort_values(ascending=False)

(321772, 49)


SO2 Mass Measure Indicator       321772
SO2 Rate Measure Indicator       321772
CO2 Mass Measure Indicator       321772
CO2 Rate Measure Indicator       321772
NOx Mass Measure Indicator       321772
NOx Rate Measure Indicator       321772
Heat Input Measure Indicator     321772
Secondary Fuel Type              309338
Planned Retirement Year          260353
SO2 Controls                     213031
NOx Phase                        200844
Nameplate Capacity (MW) - EIA     91650
Summer Capacity (MW)              91650
Winter Capacity (MW)              91650
Minimum Load (MW)                 91650
NOx Controls                       9375
dtype: int64

SO2 Mass Measure Indicator       100.000000
SO2 Rate Measure Indicator       100.000000
CO2 Mass Measure Indicator       100.000000
CO2 Rate Measure Indicator       100.000000
NOx Mass Measure Indicator       100.000000
NOx Rate Measure Indicator       100.000000
Heat Input Measure Indicator     100.000000
Secondary Fuel Type               96.135773
Planned Retirement Year           80.912261
SO2 Controls                      66.205574
NOx Phase                         62.418110
Nameplate Capacity (MW) - EIA     28.482901
Summer Capacity (MW)              28.482901
Winter Capacity (MW)              28.482901
Minimum Load (MW)                 28.482901
NOx Controls                       2.913554
dtype: float64

In [ ]:
# handle null values appropriately

# remove Steam Load columns (always null)
complete_campd_wy = complete_campd_wy.drop(columns=["Steam Load (1000 lb/hr)"])

# we drop nameplate capacity (MW) - EIA (redundant with Nameplate Capacity (MWe) and has many nulls)
complete_campd_wy = complete_campd_wy.drop(columns=["Nameplate Capacity (MW) - EIA"])

# replace nan values in categorical columns with "Nothing"
cat_cols_with_na = ["Secondary Fuel Type", "NOx Phase", 
                    "SO2 Controls", "NOx Controls",
                    "SO2 Mass Measure Indicator",
                    "SO2 Rate Measure Indicator",
                    "CO2 Mass Measure Indicator",
                    "CO2 Rate Measure Indicator",
                    "NOx Mass Measure Indicator",
                    "NOx Rate Measure Indicator",
                    "Heat Input Measure Indicator"
]
for col in cat_cols_with_na:
    complete_campd_wy[col] = complete_campd_wy[col].fillna("Nothing").astype("string")

print("Null values in complete_campd_wy after adjustments:")
null_wy = complete_campd_wy.isnull().sum()
print(null_wy[null_wy > 0].sort_values(ascending=False))

# we keep planned retirement year null values as it is a numeric col which ml models should be able to handle (if not we edit later)
# also doesn't make sense to impute planned retirement year if there is none (not a case of missing data)

Null values in complete_campd_wy after adjustments:
Planned Retirement Year    260353
Summer Capacity (MW)        91650
Winter Capacity (MW)        91650
Minimum Load (MW)           91650
dtype: int64


In [23]:
# measurement methods
print("Measurement methods in complete_campd_wy:")
display(complete_campd_wy.groupby("CO2 Mass Measure Indicator")["CO2 Mass (short tons)"].count())
display(complete_campd_wy.groupby("NOx Mass Measure Indicator")["NOx Mass Measure Indicator"].count())
display(complete_campd_wy.groupby("SO2 Mass Measure Indicator")["SO2 Mass Measure Indicator"].count())


Measurement methods in complete_campd_wy:


CO2 Mass Measure Indicator
Nothing    321772
Name: CO2 Mass (short tons), dtype: int64

NOx Mass Measure Indicator
Nothing    321772
Name: NOx Mass Measure Indicator, dtype: int64

SO2 Mass Measure Indicator
Nothing    321772
Name: SO2 Mass Measure Indicator, dtype: int64

In [24]:
# calculate hourly capacity factor
complete_campd_wy["hourly_capacity_factor"] = 0

complete_campd_wy.head(5)


,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),SO2 Mass (lbs),SO2 Mass Measure Indicator,...,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor
0,WY,Dave Johnston,4158,BW41,2019-02-18,11,0.0,0.0,0.0,Nothing,...,93.0,99.0,45.0,<NA>,60,60,60.0,2,2,0
1,WY,Dave Johnston,4158,BW41,2019-02-18,12,0.0,0.0,0.0,Nothing,...,93.0,99.0,45.0,<NA>,60,60,60.0,2,2,0
2,WY,Dave Johnston,4158,BW41,2019-02-18,13,0.0,0.0,0.0,Nothing,...,93.0,99.0,45.0,<NA>,60,60,60.0,2,2,0
3,WY,Dave Johnston,4158,BW41,2019-02-18,14,0.0,0.0,0.0,Nothing,...,93.0,99.0,45.0,<NA>,60,60,60.0,2,2,0
4,WY,Dave Johnston,4158,BW41,2019-02-18,15,0.0,0.0,0.0,Nothing,...,93.0,99.0,45.0,<NA>,60,60,60.0,2,2,0


In [ ]:
# calculate hourly ramping factor
filtered_out_rows_campd_wy = pd.read_csv('WY/Hourly/Concatted File/wy_2019_2024_Operating.csv', dtype=campd_type_dict, parse_dates=["Date"])
filtered_out_rows_campd_wy = filtered_out_rows_campd_wy[["Facility ID", "Unit ID", "Date", "Hour", "Gross Load (MW)"]]
filtered_out_rows_campd_wy["Year"] = filtered_out_rows_campd_wy["Date"].dt.year
facility_small = pd.read_csv('Facility/Concatted File/facility_2014_2024.csv', dtype={"Facility ID": "Int64", "Unit ID": "string", "Year": "Int64"})
facility_small = facility_small[["Year", 'Facility ID', 'Unit ID', "Nameplate Capacity (MWe)"]]
all_hours_wy = filtered_out_rows_campd_wy.merge(facility_small, on=['Year', 'Facility ID', 'Unit ID'], how='left')
all_hours_wy["Gross Load (MW)"] = all_hours_wy["Gross Load (MW)"].fillna(0)

all_hours_wy = pd.concat([all_hours_wy, complete_campd_wy[["Facility ID", "Unit ID", "Date", "Hour", "Gross Load (MW)", "Year", "Nameplate Capacity (MWe)"]]], ignore_index=True)

# Sort data by unit and time to calculate t-1 values
all_hours_wy = all_hours_wy.sort_values(["Facility ID", "Unit ID", "Date", "Hour"])

# Calculate lagged gross load per unit (get previous row's value for gross load)
all_hours_wy["gross_load_lag1"] = all_hours_wy.groupby(["Facility ID", "Unit ID"])["Gross Load (MW)"].shift(1)

# calc ramping factor
all_hours_wy["hourly_ramping_factor"] = (
    all_hours_wy["Gross Load (MW)"] - all_hours_wy["gross_load_lag1"]
) / all_hours_wy["Nameplate Capacity (MWe)"]

display(all_hours_wy.head(10))

complete_campd_wy = complete_campd_wy.merge(all_hours_wy[["Facility ID", "Unit ID", "Date", "Hour", "hourly_ramping_factor"]],
                       on=["Facility ID", "Unit ID", "Date", "Hour"],
                       how="left")
complete_campd_wy.head(5)


,Facility ID,Unit ID,Date,Hour,Gross Load (MW),Year,Nameplate Capacity (MWe),gross_load_lag1,hourly_ramping_factor
0,4158,BW41,2019-01-01,0,103.0,2019,106.0,NaN,NaN
1,4158,BW41,2019-01-01,1,103.0,2019,106.0,103.0,0.000000
2,4158,BW41,2019-01-01,2,103.0,2019,106.0,103.0,0.000000
3,4158,BW41,2019-01-01,3,104.0,2019,106.0,103.0,0.009434
4,4158,BW41,2019-01-01,4,103.0,2019,106.0,104.0,-0.009434
5,4158,BW41,2019-01-01,5,103.0,2019,106.0,103.0,0.000000
6,4158,BW41,2019-01-01,6,103.0,2019,106.0,103.0,0.000000
7,4158,BW41,2019-01-01,7,104.0,2019,106.0,103.0,0.009434
8,4158,BW41,2019-01-01,8,104.0,2019,106.0,104.0,0.000000
9,4158,BW41,2019-01-01,9,103.0,2019,106.0,104.0,-0.009434


,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),SO2 Mass (lbs),SO2 Mass Measure Indicator,...,Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor,hourly_ramping_factor
0,WY,Dave Johnston,4158,BW41,2019-02-18,11,0.0,0.0,0.0,Nothing,...,99.0,45.0,<NA>,60,60,60.0,2,2,0,-0.981132
1,WY,Dave Johnston,4158,BW41,2019-02-18,12,0.0,0.0,0.0,Nothing,...,99.0,45.0,<NA>,60,60,60.0,2,2,0,0.000000
2,WY,Dave Johnston,4158,BW41,2019-02-18,13,0.0,0.0,0.0,Nothing,...,99.0,45.0,<NA>,60,60,60.0,2,2,0,0.000000
3,WY,Dave Johnston,4158,BW41,2019-02-18,14,0.0,0.0,0.0,Nothing,...,99.0,45.0,<NA>,60,60,60.0,2,2,0,0.000000
4,WY,Dave Johnston,4158,BW41,2019-02-18,15,0.0,0.0,0.0,Nothing,...,99.0,45.0,<NA>,60,60,60.0,2,2,0,0.000000


In [32]:
complete_campd_wy.to_csv('WY/Hourly/Concatted File/wy_2019_2024_complete_NonOperating.csv', index=False)

<h3>New York</h3>

In [39]:
complete_campd_ny = pd.read_csv(
    'NY/Hourly/Concatted File/ny_2019_2024_with_manualEIA_NonOperating.csv',  
    usecols=complete_campd_columns,
    dtype=complete_campd_type_dict,
    parse_dates=["Date"]
)

# extract day and month as new col from date
complete_campd_ny["Month"] = complete_campd_ny["Date"].dt.month
complete_campd_ny["Day"] = complete_campd_ny["Date"].dt.month


display(complete_campd_ny.head())
print(complete_campd_ny.dtypes)

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Nameplate Capacity (MW) - EIA,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day
0,NY,Danskammer Generating Station,2480,1,2019-01-01,0,0.0,0.0,0.0,0.0,...,72.0,70.7,68.2,20.0,<NA>,68,68,68.0,1,1
1,NY,Danskammer Generating Station,2480,1,2019-01-01,1,0.0,0.0,0.0,0.0,...,72.0,70.7,68.2,20.0,<NA>,68,68,68.0,1,1
2,NY,Danskammer Generating Station,2480,1,2019-01-01,2,0.0,0.0,0.0,0.0,...,72.0,70.7,68.2,20.0,<NA>,68,68,68.0,1,1
3,NY,Danskammer Generating Station,2480,1,2019-01-01,3,0.0,0.0,0.0,0.0,...,72.0,70.7,68.2,20.0,<NA>,68,68,68.0,1,1
4,NY,Danskammer Generating Station,2480,1,2019-01-01,4,0.0,0.0,0.0,0.0,...,72.0,70.7,68.2,20.0,<NA>,68,68,68.0,1,1


State                                               string[python]
Facility Name                                       string[python]
Facility ID                                                  Int64
Unit ID                                             string[python]
Date                                                datetime64[ns]
Hour                                                         Int64
Operating Time                                             float64
Gross Load (MW)                                            float64
Steam Load (1000 lb/hr)                                    float64
SO2 Mass (lbs)                                             float64
SO2 Mass Measure Indicator                          string[python]
SO2 Rate (lbs/mmBtu)                                       float64
SO2 Rate Measure Indicator                          string[python]
CO2 Mass (short tons)                                      float64
CO2 Mass Measure Indicator                          string[pyt

In [34]:
print(complete_campd_ny.shape)
null_ny = complete_campd_ny.isnull().sum()
null_ny = null_ny[null_ny > 0].sort_values(ascending=False)
display(null_ny)

# Give null vals as percentage of total rows
total_rows = complete_campd_ny.shape[0]
null_percentages = (null_ny / total_rows) * 100
null_percentages.sort_values(ascending=False)

(11627974, 49)


SO2 Mass Measure Indicator                          11627974
SO2 Rate Measure Indicator                          11627974
CO2 Mass Measure Indicator                          11627974
CO2 Rate Measure Indicator                          11627974
NOx Mass Measure Indicator                          11627974
NOx Rate Measure Indicator                          11627974
Heat Input Measure Indicator                        11627974
Planned Retirement Year                             11575536
SO2 Controls                                        11516501
NOx Phase                                           11299789
SO2 Phase                                            7582149
NOx Controls                                         5953276
Nameplate Capacity (MW) - EIA                        5232542
Minimum Load (MW)                                    5232542
Winter Capacity (MW)                                 5232542
Summer Capacity (MW)                                 5232542
unit_age_mean           

SO2 Mass Measure Indicator                          100.000000
SO2 Rate Measure Indicator                          100.000000
CO2 Mass Measure Indicator                          100.000000
CO2 Rate Measure Indicator                          100.000000
NOx Mass Measure Indicator                          100.000000
NOx Rate Measure Indicator                          100.000000
Heat Input Measure Indicator                        100.000000
Planned Retirement Year                              99.549036
SO2 Controls                                         99.041338
NOx Phase                                            97.177625
SO2 Phase                                            65.206106
NOx Controls                                         51.197878
Summer Capacity (MW)                                 44.999602
Winter Capacity (MW)                                 44.999602
Minimum Load (MW)                                    44.999602
Nameplate Capacity (MW) - EIA                        44

In [ ]:
# handle categorical null values appropriately
cat_cols_with_na_none = [
    "SO2 Controls",
    "NOx Phase",
    "Secondary Fuel Type",
    "NOx Controls",
    "SO2 Phase",
    "CO2 Rate Measure Indicator",
    "SO2 Mass Measure Indicator",
    "SO2 Rate Measure Indicator",
    "CO2 Mass Measure Indicator",
    "NOx Mass Measure Indicator",
    "NOx Rate Measure Indicator",
    "Heat Input Measure Indicator"
]

cat_cols_with_na_unknown = [
    "Operating Status",
    "Associated Generators & Nameplate Capacity (MWe)",
    "Source Category", "Program Code"
]

# here nan means that no data was likely purposely reported -> fill with "None"
for col in cat_cols_with_na_none:
    complete_campd_ny[col] = complete_campd_ny[col].fillna("Nothing").astype("string")

# here nan means that no data was likely reported -> fill with "Missing"
for col in cat_cols_with_na_unknown:
    complete_campd_ny[col] = complete_campd_ny[col].fillna("Missing").astype("string")

num_cols_with_na = [
    "Planned Retirement Year",
    "Steam Load (1000 lb/hr)",
    "unit_age_max",
    "unit_age_min",
    "unit_age_mean",
    "Minimum Load (MW)",
    "Winter Capacity (MW)",
    "Summer Capacity (MW)",
    #"Nameplate Capacity (MW) - EIA", # dropped later
    "Gross Load (MW)",
    "Longitude",
    "Latitude",
    "Max Hourly HI Rate (mmBtu/hr)",
    "Nameplate Capacity (MWe)",
    "CO2 Rate (short tons/mmBtu)",
    "SO2 Rate (lbs/mmBtu)",
    "NOx Rate (lbs/mmBtu)"
]

# we drop nameplate capacity (MW) - EIA (redundant with Nameplate Capacity (MWe) and has many nulls)
complete_campd_ny = complete_campd_ny.drop(columns=["Nameplate Capacity (MW) - EIA"])

# for now we keep it, adjust for models later
print("Null value percentages in complete_campd_ny after adjustments:")
total_rows = complete_campd_ny.shape[0]
null_ny = complete_campd_ny.isnull().sum()
null_ny = null_ny[null_ny > 0].sort_values(ascending=False)
null_percentages = (null_ny / total_rows) * 100
null_percentages.sort_values(ascending=False)



Null value percentages in complete_campd_ny after adjustments:


Planned Retirement Year          99.549036
Summer Capacity (MW)             44.999602
Winter Capacity (MW)             44.999602
Minimum Load (MW)                44.999602
unit_age_min                     43.822501
unit_age_max                     43.822501
unit_age_mean                    43.822501
Latitude                         27.657535
Longitude                        27.657535
Max Hourly HI Rate (mmBtu/hr)    27.657535
Nameplate Capacity (MWe)         27.657535
dtype: float64

In [41]:
# measurement methods
print("Measurement methods in complete_campd_ny:")
display(complete_campd_ny.groupby("CO2 Mass Measure Indicator")["CO2 Mass (short tons)"].count())
display(complete_campd_ny.groupby("NOx Mass Measure Indicator")["NOx Mass Measure Indicator"].count())
display(complete_campd_ny.groupby("SO2 Mass Measure Indicator")["SO2 Mass Measure Indicator"].count())


Measurement methods in complete_campd_ny:


CO2 Mass Measure Indicator
Nothing    11627974
Name: CO2 Mass (short tons), dtype: int64

NOx Mass Measure Indicator
Nothing    11627974
Name: NOx Mass Measure Indicator, dtype: int64

SO2 Mass Measure Indicator
Nothing    11627974
Name: SO2 Mass Measure Indicator, dtype: int64

In [42]:
# calculate hourly capacity factor
complete_campd_ny["hourly_capacity_factor"] = 0

complete_campd_ny.head(5)

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor
0,NY,Danskammer Generating Station,2480,1,2019-01-01,0,0.0,0.0,0.0,0.0,...,70.7,68.2,20.0,<NA>,68,68,68.0,1,1,0
1,NY,Danskammer Generating Station,2480,1,2019-01-01,1,0.0,0.0,0.0,0.0,...,70.7,68.2,20.0,<NA>,68,68,68.0,1,1,0
2,NY,Danskammer Generating Station,2480,1,2019-01-01,2,0.0,0.0,0.0,0.0,...,70.7,68.2,20.0,<NA>,68,68,68.0,1,1,0
3,NY,Danskammer Generating Station,2480,1,2019-01-01,3,0.0,0.0,0.0,0.0,...,70.7,68.2,20.0,<NA>,68,68,68.0,1,1,0
4,NY,Danskammer Generating Station,2480,1,2019-01-01,4,0.0,0.0,0.0,0.0,...,70.7,68.2,20.0,<NA>,68,68,68.0,1,1,0


In [ ]:
# calculate hourly ramping factor
filtered_out_rows_campd_ny = pd.read_csv('NY/Hourly/Concatted File/ny_2019_2024_Operating.csv', dtype=campd_type_dict, parse_dates=["Date"])
filtered_out_rows_campd_ny["Year"] = filtered_out_rows_campd_ny["Date"].dt.year
facility_small = pd.read_csv('Facility/Concatted File/facility_2014_2024.csv', dtype={"Facility ID": "Int64", "Unit ID": "string", "Year": "Int64"})
facility_small = facility_small[["Year", 'Facility ID', 'Unit ID', "Nameplate Capacity (MWe)"]]
all_hours_ny = filtered_out_rows_campd_ny.merge(facility_small, on=['Year', 'Facility ID', 'Unit ID'], how='left')
all_hours_ny["Gross Load (MW)"] = all_hours_ny["Gross Load (MW)"].fillna(0)

all_hours_ny = pd.concat([all_hours_ny, complete_campd_ny[["Facility ID", "Unit ID", "Date", "Hour", "Gross Load (MW)", "Year", "Nameplate Capacity (MWe)"]]], ignore_index=True)

# Sort data by unit and time to calculate t-1 values
all_hours_ny = all_hours_ny.sort_values(["Facility ID", "Unit ID", "Date", "Hour"])

# Calculate lagged gross load per unit (get previous row's value for gross load)
all_hours_ny["gross_load_lag1"] = all_hours_ny.groupby(["Facility ID", "Unit ID"])["Gross Load (MW)"].shift(1)

# calc ramping factor
all_hours_ny["hourly_ramping_factor"] = (
    all_hours_ny["Gross Load (MW)"] - all_hours_ny["gross_load_lag1"]
) / all_hours_ny["Nameplate Capacity (MWe)"]

display(all_hours_ny.head(10))

complete_campd_ny = complete_campd_ny.merge(all_hours_ny[["Facility ID", "Unit ID", "Date", "Hour", "hourly_ramping_factor"]],
                        on=["Facility ID", "Unit ID", "Date", "Hour"],
                        how="left")
complete_campd_ny.head(5)


,State,Facility Name,Facility ID,Unit ID,Associated Stacks,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),...,Unit Type,SO2 Controls,NOx Controls,PM Controls,Hg Controls,Program Code,Year,Nameplate Capacity (MWe),gross_load_lag1,hourly_ramping_factor
2400701,<NA>,<NA>,2480,1,<NA>,2019-01-01,0,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,72.0,NaN,NaN
2400702,<NA>,<NA>,2480,1,<NA>,2019-01-01,1,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,72.0,0.0,0.0
2400703,<NA>,<NA>,2480,1,<NA>,2019-01-01,2,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,72.0,0.0,0.0
2400704,<NA>,<NA>,2480,1,<NA>,2019-01-01,3,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,72.0,0.0,0.0
2400705,<NA>,<NA>,2480,1,<NA>,2019-01-01,4,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,72.0,0.0,0.0
2400706,<NA>,<NA>,2480,1,<NA>,2019-01-01,5,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,72.0,0.0,0.0
2400707,<NA>,<NA>,2480,1,<NA>,2019-01-01,6,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,72.0,0.0,0.0
2400708,<NA>,<NA>,2480,1,<NA>,2019-01-01,7,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,72.0,0.0,0.0
2400709,<NA>,<NA>,2480,1,<NA>,2019-01-01,8,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,72.0,0.0,0.0
2400710,<NA>,<NA>,2480,1,<NA>,2019-01-01,9,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,72.0,0.0,0.0


,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor,hourly_ramping_factor
0,NY,Danskammer Generating Station,2480,1,2019-01-01,0,0.0,0.0,0.0,0.0,...,68.2,20.0,<NA>,68,68,68.0,1,1,0,NaN
1,NY,Danskammer Generating Station,2480,1,2019-01-01,1,0.0,0.0,0.0,0.0,...,68.2,20.0,<NA>,68,68,68.0,1,1,0,0.0
2,NY,Danskammer Generating Station,2480,1,2019-01-01,2,0.0,0.0,0.0,0.0,...,68.2,20.0,<NA>,68,68,68.0,1,1,0,0.0
3,NY,Danskammer Generating Station,2480,1,2019-01-01,3,0.0,0.0,0.0,0.0,...,68.2,20.0,<NA>,68,68,68.0,1,1,0,0.0
4,NY,Danskammer Generating Station,2480,1,2019-01-01,4,0.0,0.0,0.0,0.0,...,68.2,20.0,<NA>,68,68,68.0,1,1,0,0.0


In [46]:
complete_campd_ny.to_csv('NY/Hourly/Concatted File/ny_2019_2024_complete_NonOperating.csv', index=False)

<h3>Texas</h3>

In [65]:
complete_campd_tx = pd.read_csv(
    'TX/Hourly/Concatted File/tx_2019_2024_with_manualEIA_NonOperating.csv',  
    usecols=complete_campd_columns,
    dtype=complete_campd_type_dict,
    parse_dates=["Date"]
)

# extract day and month as new col from date
complete_campd_tx["Month"] = complete_campd_tx["Date"].dt.month
complete_campd_tx["Day"] = complete_campd_tx["Date"].dt.day

display(complete_campd_tx.head())
print(complete_campd_tx.dtypes)

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Nameplate Capacity (MW) - EIA,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day
0,TX,Copper Station,9,CTG-1,2019-01-01,0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1
1,TX,Copper Station,9,CTG-1,2019-01-01,1,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1
2,TX,Copper Station,9,CTG-1,2019-01-01,2,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1
3,TX,Copper Station,9,CTG-1,2019-01-01,3,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1
4,TX,Copper Station,9,CTG-1,2019-01-01,4,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1


State                                               string[python]
Facility Name                                       string[python]
Facility ID                                                  Int64
Unit ID                                             string[python]
Date                                                datetime64[ns]
Hour                                                         Int64
Operating Time                                             float64
Gross Load (MW)                                            float64
Steam Load (1000 lb/hr)                                    float64
SO2 Mass (lbs)                                             float64
SO2 Mass Measure Indicator                          string[python]
SO2 Rate (lbs/mmBtu)                                       float64
SO2 Rate Measure Indicator                          string[python]
CO2 Mass (short tons)                                      float64
CO2 Mass Measure Indicator                          string[pyt

In [66]:
print(complete_campd_tx.shape)
null_tx = complete_campd_tx.isnull().sum()
null_tx = null_tx[null_tx > 0].sort_values(ascending=False)
display(null_tx)

# Give null vals as percentage of total rows
total_rows = complete_campd_tx.shape[0]
null_percentages = (null_tx / total_rows) * 100

null_percentages.sort_values(ascending=False)

(11224101, 49)


SO2 Mass Measure Indicator                          11224101
SO2 Rate Measure Indicator                          11224101
CO2 Mass Measure Indicator                          11224101
CO2 Rate Measure Indicator                          11224101
NOx Mass Measure Indicator                          11224101
NOx Rate Measure Indicator                          11224101
Heat Input Measure Indicator                        11224101
SO2 Controls                                        11029992
NOx Phase                                           10863356
Planned Retirement Year                             10258656
Secondary Fuel Type                                  7544495
Nameplate Capacity (MW) - EIA                        4085304
Minimum Load (MW)                                    4085304
Winter Capacity (MW)                                 4085304
Summer Capacity (MW)                                 4085304
unit_age_max                                         3591329
unit_age_min            

SO2 Mass Measure Indicator                          100.000000
CO2 Mass Measure Indicator                          100.000000
CO2 Rate Measure Indicator                          100.000000
NOx Mass Measure Indicator                          100.000000
NOx Rate Measure Indicator                          100.000000
Heat Input Measure Indicator                        100.000000
SO2 Rate Measure Indicator                          100.000000
SO2 Controls                                         98.270605
NOx Phase                                            96.785979
Planned Retirement Year                              91.398465
Secondary Fuel Type                                  67.216920
Winter Capacity (MW)                                 36.397605
Summer Capacity (MW)                                 36.397605
Minimum Load (MW)                                    36.397605
Nameplate Capacity (MW) - EIA                        36.397605
unit_age_max                                         31

In [67]:
# handle categorical null values appropriately

cat_cols_with_na_none = [
    "SO2 Controls",
    "NOx Phase",
    "Secondary Fuel Type",
    "NOx Controls",
    "SO2 Phase", "SO2 Mass Measure Indicator",
    "CO2 Mass Measure Indicator", "NOx Mass Measure Indicator",
    "Heat Input Measure Indicator",
    "CO2 Rate Measure Indicator",
    "SO2 Mass Measure Indicator",
    "SO2 Rate Measure Indicator",
    "NOx Rate Measure Indicator",
    "Heat Input Measure Indicator"
]

cat_cols_with_na_unknown = [
    "Operating Status",
    "Associated Generators & Nameplate Capacity (MWe)",
    "Source Category",
    "CO2 Rate Measure Indicator",
    "SO2 Rate Measure Indicator",
    "NOx Rate Measure Indicator", "Program Code"
]

# here nan means that no data was likely purposely reported -> fill with "Nothing"
for col in cat_cols_with_na_none:
    complete_campd_tx[col] = complete_campd_tx[col].fillna("Nothing").astype("string")

# here nan means that no data was likely reported -> fill with "Missing"
for col in cat_cols_with_na_unknown:
    complete_campd_tx[col] = complete_campd_tx[col].fillna("Missing").astype("string")

num_cols_with_na = [
    "Planned Retirement Year",
    "unit_age_max",
    "unit_age_min",
    "unit_age_mean",
    "Minimum Load (MW)",
    "Winter Capacity (MW)",
    "Summer Capacity (MW)",
    #"Nameplate Capacity (MW) - EIA", # dropped later
    "Longitude",
    "Latitude",
    "Max Hourly HI Rate (mmBtu/hr)",
    "Nameplate Capacity (MWe)"
]

# we drop nameplate capacity (MW) - EIA (redundant with Nameplate Capacity (MWe) and has many nulls)
complete_campd_tx = complete_campd_tx.drop(columns=["Nameplate Capacity (MW) - EIA"])

print("Null value percentages in complete_campd_tx after adjustments:")
total_rows = complete_campd_tx.shape[0]
null_tx = complete_campd_tx.isnull().sum()
null_tx = null_tx[null_tx > 0].sort_values(ascending=False)
null_percentages = (null_tx / total_rows) * 100
null_percentages.sort_values(ascending=False)

Null value percentages in complete_campd_tx after adjustments:


Planned Retirement Year          91.398465
Summer Capacity (MW)             36.397605
Winter Capacity (MW)             36.397605
Minimum Load (MW)                36.397605
unit_age_min                     31.996585
unit_age_max                     31.996585
unit_age_mean                    31.996585
Latitude                          5.011920
Longitude                         5.011920
Max Hourly HI Rate (mmBtu/hr)     5.011920
Nameplate Capacity (MWe)          5.011920
dtype: float64

In [68]:
# measurement methods
print("Measurement methods in complete_campd_tx:")
display(complete_campd_tx.groupby("CO2 Mass Measure Indicator")["CO2 Mass (short tons)"].count())
display(complete_campd_tx.groupby("NOx Mass Measure Indicator")["NOx Mass Measure Indicator"].count())
display(complete_campd_tx.groupby("SO2 Mass Measure Indicator")["SO2 Mass Measure Indicator"].count())

Measurement methods in complete_campd_tx:


CO2 Mass Measure Indicator
Nothing    11224101
Name: CO2 Mass (short tons), dtype: int64

NOx Mass Measure Indicator
Nothing    11224101
Name: NOx Mass Measure Indicator, dtype: int64

SO2 Mass Measure Indicator
Nothing    11224101
Name: SO2 Mass Measure Indicator, dtype: int64

In [69]:
# set hourly capacity factor (0 here since no gross load)
complete_campd_tx["hourly_capacity_factor"] = 0
complete_campd_tx.head(5)

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor
0,TX,Copper Station,9,CTG-1,2019-01-01,0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,0
1,TX,Copper Station,9,CTG-1,2019-01-01,1,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,0
2,TX,Copper Station,9,CTG-1,2019-01-01,2,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,0
3,TX,Copper Station,9,CTG-1,2019-01-01,3,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,0
4,TX,Copper Station,9,CTG-1,2019-01-01,4,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,0


In [ ]:
# calculate hourly ramping factor
filtered_out_rows_campd_tx = pd.read_csv('TX/Hourly/Concatted File/tx_2019_2024_Operating.csv', dtype=campd_type_dict, parse_dates=["Date"])
filtered_out_rows_campd_tx["Year"] = filtered_out_rows_campd_tx["Date"].dt.year
facility_small = pd.read_csv('Facility/Concatted File/facility_2014_2024.csv', dtype={"Facility ID": "Int64", "Unit ID": "string", "Year": "Int64"})
facility_small = facility_small[["Year", 'Facility ID', 'Unit ID', "Nameplate Capacity (MWe)"]]
all_hours_tx = filtered_out_rows_campd_tx.merge(facility_small, on=['Year', 'Facility ID', 'Unit ID'], how='left')
all_hours_tx["Gross Load (MW)"] = all_hours_tx["Gross Load (MW)"].fillna(0)

all_hours_tx = pd.concat([all_hours_tx, complete_campd_tx[["Facility ID", "Unit ID", "Date", "Hour", "Gross Load (MW)", "Year", "Nameplate Capacity (MWe)"]]], ignore_index=True)

# Sort data by unit and time to calculate t-1 values
all_hours_tx = all_hours_tx.sort_values(["Facility ID", "Unit ID", "Date", "Hour"])

# Calculate lagged gross load per unit (get previous row's value for gross load)
all_hours_tx["gross_load_lag1"] = all_hours_tx.groupby(["Facility ID", "Unit ID"])["Gross Load (MW)"].shift(1)

# calc ramping factor
all_hours_tx["hourly_ramping_factor"] = (
    all_hours_tx["Gross Load (MW)"] - all_hours_tx["gross_load_lag1"]
) / all_hours_tx["Nameplate Capacity (MWe)"]

display(all_hours_tx.head(10))

complete_campd_tx = complete_campd_tx.merge(all_hours_tx[["Facility ID", "Unit ID", "Date", "Hour", "hourly_ramping_factor"]],
                        on=["Facility ID", "Unit ID", "Date", "Hour"],
                        how="left")
complete_campd_tx.head(5)

,State,Facility Name,Facility ID,Unit ID,Associated Stacks,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),...,Unit Type,SO2 Controls,NOx Controls,PM Controls,Hg Controls,Program Code,Year,Nameplate Capacity (MWe),gross_load_lag1,hourly_ramping_factor
9262325,<NA>,<NA>,9,CTG-1,<NA>,2019-01-01,0,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,NaN,NaN,NaN
9262326,<NA>,<NA>,9,CTG-1,<NA>,2019-01-01,1,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,NaN,0.0,NaN
9262327,<NA>,<NA>,9,CTG-1,<NA>,2019-01-01,2,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,NaN,0.0,NaN
9262328,<NA>,<NA>,9,CTG-1,<NA>,2019-01-01,3,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,NaN,0.0,NaN
9262329,<NA>,<NA>,9,CTG-1,<NA>,2019-01-01,4,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,NaN,0.0,NaN
9262330,<NA>,<NA>,9,CTG-1,<NA>,2019-01-01,5,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,NaN,0.0,NaN
9262331,<NA>,<NA>,9,CTG-1,<NA>,2019-01-01,6,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,NaN,0.0,NaN
9262332,<NA>,<NA>,9,CTG-1,<NA>,2019-01-01,7,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,NaN,0.0,NaN
9262333,<NA>,<NA>,9,CTG-1,<NA>,2019-01-01,18,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,NaN,0.0,NaN
9262334,<NA>,<NA>,9,CTG-1,<NA>,2019-01-01,19,NaN,0.0,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2019,NaN,0.0,NaN


,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor,hourly_ramping_factor
0,TX,Copper Station,9,CTG-1,2019-01-01,0,0.0,0.0,0.0,0.0,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,0,NaN
1,TX,Copper Station,9,CTG-1,2019-01-01,1,0.0,0.0,0.0,0.0,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,0,NaN
2,TX,Copper Station,9,CTG-1,2019-01-01,2,0.0,0.0,0.0,0.0,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,0,NaN
3,TX,Copper Station,9,CTG-1,2019-01-01,3,0.0,0.0,0.0,0.0,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,0,NaN
4,TX,Copper Station,9,CTG-1,2019-01-01,4,0.0,0.0,0.0,0.0,...,NaN,NaN,<NA>,<NA>,<NA>,NaN,1,1,0,NaN


In [75]:
# note the nan values here are due to missing nameplate capacities

In [76]:
complete_campd_tx.to_csv('TX/Hourly/Concatted File/tx_2019_2024_complete_NonOperating.csv', index=False)

<h2>Transform and merge EIA-860 Data using crosswalk</h2>

In [3]:
# read eia generator data
eia_generator = pd.read_csv('EIA/Concatted File/eia_generator_2019_2023.csv')

# these cols have blank strings (" ") that should be nan
columns_to_clean = [
    "Nameplate Power Factor", "Summer Capacity (MW)", "Winter Capacity (MW)",
    "Minimum Load (MW)", "Planned Retirement Year"
]

# replace blank strings with nan (any strings with only whitespace)
eia_generator[columns_to_clean] = eia_generator[columns_to_clean].replace(r'^\s*$', np.nan, regex=True)

# now define full dtype dictionary
eia_generator_type_dict = {
    "Year": "Int64", 
    "Utility ID": "string",
    "Utility Name": "string",
    "Plant Code": "Int64",  
    "Plant Name": "string",
    "State": "string",
    "Generator ID": "string",
    "Technology": "string",
    "Prime Mover": "string",
    "Nameplate Capacity (MW)": "float64",
    "Nameplate Power Factor": "float64", 
    "Summer Capacity (MW)": "float64",  
    "Winter Capacity (MW)": "float64",
    "Minimum Load (MW)": "float64",
    "Operating Year": "Int64",
    "Planned Retirement Year": "float64"
}

# filter needed cols
eia_generator = eia_generator[["Year","Utility ID","Utility Name", "Plant Code", "Plant Name", "State", "Generator ID", "Technology", "Prime Mover", "Nameplate Capacity (MW)", "Nameplate Power Factor", "Summer Capacity (MW)", "Winter Capacity (MW)", "Minimum Load (MW)", "Operating Year", "Planned Retirement Year"]]

# Artificially extend the unit_level dataframe to include 2024 data (just copy the 2023 data and set Year to 2024)
eia_generator_2023 = eia_generator[eia_generator["Year"] == 2023].copy()
eia_generator_2024 = eia_generator_2023.copy()
eia_generator_2024["Year"] = 2024
eia_generator = pd.concat([eia_generator, eia_generator_2024], ignore_index=True)

# cast all dtypes according to the dictionary
eia_generator = eia_generator.astype(eia_generator_type_dict)
eia_generator


,Year,Utility ID,Utility Name,Plant Code,Plant Name,State,Generator ID,Technology,Prime Mover,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year,Planned Retirement Year
0,2019,15474,Public Service Co of Oklahoma,127,Oklaunion,TX,1,Conventional Steam Coal,ST,720.0,0.90,650.0,650.0,215.0,1986,2020.0
1,2019,54888,NRG Texas Power LLC,298,Limestone,TX,1,Conventional Steam Coal,ST,893.0,0.94,831.0,831.0,300.0,1985,2030.0
2,2019,54888,NRG Texas Power LLC,298,Limestone,TX,2,Conventional Steam Coal,ST,956.8,0.92,858.0,858.0,300.0,1986,2030.0
3,2019,58971,Danskammer Energy,2480,Danskammer Generating Station,NY,1,Natural Gas Steam Turbine,ST,72.0,0.85,70.7,68.2,20.0,1951,NaN
4,2019,58971,Danskammer Energy,2480,Danskammer Generating Station,NY,2,Natural Gas Steam Turbine,ST,73.5,0.85,65.7,65.5,20.0,1954,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4181,2024,63082,ProEnergy Services,65373,Brotman Power Station,TX,CTG-2,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,44.6,49.8,22.5,2023,NaN
4182,2024,63082,ProEnergy Services,65373,Brotman Power Station,TX,CTG-3,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,44.6,49.8,22.5,2023,NaN
4183,2024,63082,ProEnergy Services,65373,Brotman Power Station,TX,CTG-4,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,44.6,49.8,22.5,2023,NaN
4184,2024,63082,ProEnergy Services,65373,Brotman Power Station,TX,CTG-5,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,44.6,49.8,22.5,2023,NaN


In [4]:
crosswalk = pd.read_csv("epa_eia_crosswalk.csv")
crosswalk = crosswalk[(crosswalk["CAMD_STATE"] == "NY") | (crosswalk["CAMD_STATE"] == "TX") | (crosswalk["CAMD_STATE"] == "WY")] # filter for the states we need
crosswalk[crosswalk["CAMD_PLANT_ID"] == 2490]

# the col "CAMD_STATUS_DATE" gives the date where changes where last made for this plant id and unit id combo. we have to create new rows for every individual year up until the next change and so on with an additional col "Year"
# to match with the campd data
# join on CAMP_PLANT_ID, CAMD_UNIT_ID and add CAMD_NAMEPLATE_CAPACITY, CAMD_STATUS_DATE, EIA_NAMEPLATE_CAPACITY, EIA_BOILER_ID

,SEQUENCE_NUMBER,CAMD_STATE,CAMD_FACILITY_NAME,CAMD_PLANT_ID,CAMD_UNIT_ID,CAMD_GENERATOR_ID,CAMD_NAMEPLATE_CAPACITY,CAMD_FUEL_TYPE,CAMD_LATITUDE,CAMD_LONGITUDE,...,EIA_LATITUDE,EIA_LONGITUDE,EIA_RETIRE_YEAR,PLANT_ID_CHANGE_FLAG,MOD_EIA_PLANT_ID,MOD_EIA_BOILER_ID,MOD_EIA_GENERATOR_ID_BOILER,MOD_EIA_GENERATOR_ID_GEN,MATCH_TYPE_GEN,MATCH_TYPE_BOILER
1772,1773,NY,Arthur Kill,2490,20,2,376.2,Pipeline Natural Gas,40.5915,-74.2027,...,40.591564,-74.200035,0.0,0.0,2490.0,20,2,2,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...
1773,1774,NY,Arthur Kill,2490,30,3,535.5,Pipeline Natural Gas,40.5915,-74.2027,...,40.591564,-74.200035,0.0,0.0,2490.0,30,3,3,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...
1774,1775,NY,Arthur Kill,2490,CT0001,GT1,16.3,Pipeline Natural Gas,40.5915,-74.2027,...,40.591564,-74.200035,0.0,0.0,2490.0,NaN,NaN,GT1,3_1_Generator (generators) match on plant and ...,NaN


In [5]:
# create yearly rows manually

crosswalk["CAMD_STATUS_DATE"] = pd.to_datetime(crosswalk["CAMD_STATUS_DATE"])

crosswalk_sorted = crosswalk.sort_values(by=["CAMD_PLANT_ID", "CAMD_UNIT_ID", "CAMD_STATUS_DATE"])

# Define end year up to which we generate rows
default_end_year = 2024

expanded_rows = [] # save rows for expansion

# Group by plant/unit and generate yearly rows
for (plant_id, unit_id), group in crosswalk_sorted.groupby(["CAMD_PLANT_ID", "CAMD_UNIT_ID"]):
    group = group.sort_values("CAMD_STATUS_DATE").reset_index(drop=True)

    for i in range(len(group)):
        start_year = group.loc[i, "CAMD_STATUS_DATE"].year
        end_year = (group.loc[i + 1, "CAMD_STATUS_DATE"].year 
                    if i + 1 < len(group) 
                    else default_end_year + 1)  # +1 so we include the final year

        for year in range(start_year, end_year):
            row = group.loc[i].copy()
            row["Year"] = year
            expanded_rows.append(row)

# Combine into new df
expanded_crosswalk = pd.DataFrame(expanded_rows)

expanded_crosswalk = expanded_crosswalk[expanded_crosswalk["Year"] >= 2019]
pd.set_option('display.max_columns', None)
expanded_crosswalk.sort_values(by=["CAMD_PLANT_ID", "CAMD_UNIT_ID", "Year"])

,SEQUENCE_NUMBER,CAMD_STATE,CAMD_FACILITY_NAME,CAMD_PLANT_ID,CAMD_UNIT_ID,CAMD_GENERATOR_ID,CAMD_NAMEPLATE_CAPACITY,CAMD_FUEL_TYPE,CAMD_LATITUDE,CAMD_LONGITUDE,CAMD_STATUS,CAMD_STATUS_DATE,CAMD_RETIRE_YEAR,MOD_CAMD_UNIT_ID,MOD_CAMD_GENERATOR_ID,EIA_STATE,EIA_PLANT_NAME,EIA_PLANT_ID,EIA_GENERATOR_ID,EIA_NAMEPLATE_CAPACITY,EIA_BOILER_ID,EIA_UNIT_TYPE,EIA_FUEL_TYPE,EIA_LATITUDE,EIA_LONGITUDE,EIA_RETIRE_YEAR,PLANT_ID_CHANGE_FLAG,MOD_EIA_PLANT_ID,MOD_EIA_BOILER_ID,MOD_EIA_GENERATOR_ID_BOILER,MOD_EIA_GENERATOR_ID_GEN,MATCH_TYPE_GEN,MATCH_TYPE_BOILER,Year
0,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9.0,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2019
0,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9.0,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2020
0,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9.0,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2021
0,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9.0,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2022
0,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9.0,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.750,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2020
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.750,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2021
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.750,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2022
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.750,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2023


In [ ]:
expanded_crosswalk["EIA_PLANT_ID"] = expanded_crosswalk["EIA_PLANT_ID"].astype("Int64")
expanded_crosswalk["EIA_GENERATOR_ID"] = expanded_crosswalk["EIA_GENERATOR_ID"].astype("string")
expanded_crosswalk["Year"] = expanded_crosswalk["Year"].astype("Int64")

In [7]:
# join crosswalk with additional eia_generator data
combined_crosswalk = expanded_crosswalk.merge(eia_generator, how="left", left_on=["EIA_PLANT_ID", "EIA_GENERATOR_ID", "Year"], right_on=["Plant Code", "Generator ID", "Year"])
combined_crosswalk

,SEQUENCE_NUMBER,CAMD_STATE,CAMD_FACILITY_NAME,CAMD_PLANT_ID,CAMD_UNIT_ID,CAMD_GENERATOR_ID,CAMD_NAMEPLATE_CAPACITY,CAMD_FUEL_TYPE,CAMD_LATITUDE,CAMD_LONGITUDE,CAMD_STATUS,CAMD_STATUS_DATE,CAMD_RETIRE_YEAR,MOD_CAMD_UNIT_ID,MOD_CAMD_GENERATOR_ID,EIA_STATE,EIA_PLANT_NAME,EIA_PLANT_ID,EIA_GENERATOR_ID,EIA_NAMEPLATE_CAPACITY,EIA_BOILER_ID,EIA_UNIT_TYPE,EIA_FUEL_TYPE,EIA_LATITUDE,EIA_LONGITUDE,EIA_RETIRE_YEAR,PLANT_ID_CHANGE_FLAG,MOD_EIA_PLANT_ID,MOD_EIA_BOILER_ID,MOD_EIA_GENERATOR_ID_BOILER,MOD_EIA_GENERATOR_ID_GEN,MATCH_TYPE_GEN,MATCH_TYPE_BOILER,Year,Utility ID,Utility Name,Plant Code,Plant Name,State,Generator ID,Technology,Prime Mover,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year,Planned Retirement Year
0,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2019,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
1,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2020,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
2,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2021,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
3,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2022,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
4,21,TX,Copper Station,9,CTG-1,1,70.0,Pipeline Natural Gas,31.7569,-106.375,OPR,1980-07-01,0,CTG-1,1,TX,Copper,9,1,86.9,NaN,GT,NG,31.7569,-106.375,0.0,0.0,9.0,NaN,NaN,1,3_1_Generator (generators) match on plant and ...,NaN,2023,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4767,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.750,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2020,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
4768,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.750,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2021,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
4769,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.750,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2022,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
4770,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.7600,-73.750,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2023,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN


In [ ]:
# read in cleaned dfs
campd_ny = pd.read_csv("NY/Hourly/Concatted File/ny_2019_2024_complete_NonOperating.csv", dtype=campd_type_dict)
campd_wy = pd.read_csv("WY/Hourly/Concatted File/wy_2019_2024_complete_NonOperating.csv", dtype=campd_type_dict)
campd_tx = pd.read_csv("TX/Hourly/Concatted File/tx_2019_2024_complete_NonOperating.csv", dtype=campd_type_dict)

In [8]:
combined_crosswalk = expanded_crosswalk.merge(eia_generator, how="right", left_on=["EIA_PLANT_ID", "EIA_GENERATOR_ID", "Year"], right_on=["Plant Code", "Generator ID", "Year"])
combined_crosswalk

,SEQUENCE_NUMBER,CAMD_STATE,CAMD_FACILITY_NAME,CAMD_PLANT_ID,CAMD_UNIT_ID,CAMD_GENERATOR_ID,CAMD_NAMEPLATE_CAPACITY,CAMD_FUEL_TYPE,CAMD_LATITUDE,CAMD_LONGITUDE,CAMD_STATUS,CAMD_STATUS_DATE,CAMD_RETIRE_YEAR,MOD_CAMD_UNIT_ID,MOD_CAMD_GENERATOR_ID,EIA_STATE,EIA_PLANT_NAME,EIA_PLANT_ID,EIA_GENERATOR_ID,EIA_NAMEPLATE_CAPACITY,EIA_BOILER_ID,EIA_UNIT_TYPE,EIA_FUEL_TYPE,EIA_LATITUDE,EIA_LONGITUDE,EIA_RETIRE_YEAR,PLANT_ID_CHANGE_FLAG,MOD_EIA_PLANT_ID,MOD_EIA_BOILER_ID,MOD_EIA_GENERATOR_ID_BOILER,MOD_EIA_GENERATOR_ID_GEN,MATCH_TYPE_GEN,MATCH_TYPE_BOILER,Year,Utility ID,Utility Name,Plant Code,Plant Name,State,Generator ID,Technology,Prime Mover,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year,Planned Retirement Year
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,15474,Public Service Co of Oklahoma,127,Oklaunion,TX,1,Conventional Steam Coal,ST,720.0,0.90,650.0,650.0,215.0,1986,2020.0
1,160.0,TX,Limestone,298.0,LIM1,1,813.4,Coal,31.4219,-96.2525,OPR,1985-12-01,0.0,LIM1,1,TX,Limestone,298,1,893.0,LIM1,ST,SUB,31.421900,-96.252500,0.0,0.0,298.0,LIM1,1,1,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2019,54888,NRG Texas Power LLC,298,Limestone,TX,1,Conventional Steam Coal,ST,893.0,0.94,831.0,831.0,300.0,1985,2030.0
2,161.0,TX,Limestone,298.0,LIM2,2,813.4,Coal,31.4219,-96.2525,OPR,1985-12-01,0.0,LIM2,2,TX,Limestone,298,2,956.8,LIM2,ST,SUB,31.421900,-96.252500,0.0,0.0,298.0,LIM2,2,2,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2019,54888,NRG Texas Power LLC,298,Limestone,TX,2,Conventional Steam Coal,ST,956.8,0.92,858.0,858.0,300.0,1986,2030.0
3,1769.0,NY,Danskammer Generating Station,2480.0,1,1,72.0,Pipeline Natural Gas,41.5730,-73.9646,OPR,2014-09-03,0.0,1,1,NY,Danskammer Generating Station,2480,1,72.0,1,ST,NG,41.571247,-73.974981,0.0,0.0,2480.0,1,1,1,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2019,58971,Danskammer Energy,2480,Danskammer Generating Station,NY,1,Natural Gas Steam Turbine,ST,72.0,0.85,70.7,68.2,20.0,1951,NaN
4,1770.0,NY,Danskammer Generating Station,2480.0,2,2,73.5,Pipeline Natural Gas,41.5730,-73.9646,OPR,2014-09-12,0.0,2,2,NY,Danskammer Generating Station,2480,2,73.5,2,ST,NG,41.571247,-73.974981,0.0,0.0,2480.0,2,2,2,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2019,58971,Danskammer Energy,2480,Danskammer Generating Station,NY,2,Natural Gas Steam Turbine,ST,73.5,0.85,65.7,65.5,20.0,1954,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4810,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,63082,ProEnergy Services,65373,Brotman Power Station,TX,CTG-2,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,44.6,49.8,22.5,2023,NaN
4811,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,63082,ProEnergy Services,65373,Brotman Power Station,TX,CTG-3,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,44.6,49.8,22.5,2023,NaN
4812,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,63082,ProEnergy Services,65373,Brotman Power Station,TX,CTG-4,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,44.6,49.8,22.5,2023,NaN
4813,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,63082,ProEnergy Services,65373,Brotman Power Station,TX,CTG-5,Natural Gas Fired Combustion Turbin

In [9]:
combined_crosswalk = combined_crosswalk[['CAMD_PLANT_ID',
       'CAMD_UNIT_ID', 'CAMD_GENERATOR_ID', 'CAMD_NAMEPLATE_CAPACITY', 'CAMD_STATUS_DATE', 'CAMD_RETIRE_YEAR', 'EIA_PLANT_ID',
       'EIA_GENERATOR_ID', 'EIA_NAMEPLATE_CAPACITY', 'EIA_BOILER_ID', 'EIA_RETIRE_YEAR', 'Year', 'Utility ID', 'Utility Name', 'Plant Code', 'Plant Name',
       'State', 'Generator ID', 'Technology', 'Prime Mover',
       'Nameplate Capacity (MW)', 'Nameplate Power Factor',
       'Summer Capacity (MW)', 'Winter Capacity (MW)', 'Minimum Load (MW)',
       'Operating Year', 'Planned Retirement Year']]

In [10]:
# bring to CAMD unit-level

agg_funcs = {
    'Nameplate Capacity (MW)': 'sum',
    'Summer Capacity (MW)': 'sum',
    'Winter Capacity (MW)': 'sum',
    'Minimum Load (MW)': 'sum',
    'Operating Year': ['min', 'max', 'mean'], # get operating year of the oldest (min) and newest (max) generator in the unit as well as the average across all gens
    'Planned Retirement Year': 'max',
}

unit_level = combined_crosswalk.groupby(['Year', 'EIA_PLANT_ID', 'CAMD_UNIT_ID']).agg(agg_funcs).reset_index()

display(unit_level)

# flatten multi-index columns (only keep the aggregation operator (min, max, mean) for Operating Year)
unit_level.columns = [
    f"{col[0]}_{col[1]}" if col[0] == "Operating Year" else col[0]
    for col in unit_level.columns.to_flat_index()
]

# set data types again for the final unit_level dataframe
unit_level = unit_level.astype({
    'Year': 'Int64',
    'EIA_PLANT_ID': 'Int64',
    'CAMD_UNIT_ID': 'string',
    'Nameplate Capacity (MW)': 'float64',
    'Summer Capacity (MW)': 'float64',
    'Winter Capacity (MW)': 'float64',
    'Minimum Load (MW)': 'float64',
    'Operating Year_min': 'Int64',
    'Operating Year_max': 'Int64',
    'Operating Year_mean': 'Int64',
    'Planned Retirement Year': 'Int64'
})

unit_level

Year EIA_PLANT_ID CAMD_UNIT_ID Nameplate Capacity (MW)  \
                                                         sum   
0     2019          298         LIM1                   893.0   
1     2019          298         LIM2                   956.8   
2     2019         2480            1                    72.0   
3     2019         2480            2                    73.5   
4     2019         2480            3                   147.1   
...    ...          ...          ...                     ...   
3195  2024        60459          CT1                    60.5   
3196  2024        60459          CT2                    60.5   
3197  2024        60460          CT1                    60.5   
3198  2024        60460          CT2                    60.5   
3199  2024        60468         CT-1                   121.5   

     Summer Capacity (MW) Winter Capacity (MW) Minimum Load (MW)  \
                      sum                  sum               sum   
0                   831.0                831.0             300.0   
1                   858.0                858.0             300.0   
2                    70.7                 68.2              20.0   
3                    65.7                 65.5              20.0   
4                   137.7                137.1              40.0   
...                   ...                  ...               ...   
3195                 43.0                 46.0              20.0   
3196                 43.0                 46.0              20.0   
3197                 43.0                 46.0              20.0   
3198                 43.0                 46.0              20.0   
3199                117.0                119.0              25.0   

     Operating Year               Planned Retirement Year  
                min   max    mean                     max  
0              1985  1985  1985.0                  2030.0  
1              1986  1986  1986.0                  2030.0  
2              1951  1951  1951.0                     NaN  
3              1954  1954  1954.0                     NaN  
4              1959  1959  1959.0                     NaN  
...             ...   ...     ...                     ...  
3195           2017  2017  2017.0                     NaN  
3196           2017  2017  2017.0                     NaN  
3197           2017  2017  2017.0                     NaN  
3198           2017  2017  2017.0                     NaN  
3199           2018  2018  2018.0                     NaN  

[3200 rows x 11 columns]

,Year,EIA_PLANT_ID,CAMD_UNIT_ID,Nameplate Capacity (MW),Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year_min,Operating Year_max,Operating Year_mean,Planned Retirement Year
0,2019,298,LIM1,893.0,831.0,831.0,300.0,1985,1985,1985,2030
1,2019,298,LIM2,956.8,858.0,858.0,300.0,1986,1986,1986,2030
2,2019,2480,1,72.0,70.7,68.2,20.0,1951,1951,1951,<NA>
3,2019,2480,2,73.5,65.7,65.5,20.0,1954,1954,1954,<NA>
4,2019,2480,3,147.1,137.7,137.1,40.0,1959,1959,1959,<NA>
...,...,...,...,...,...,...,...,...,...,...,...
3195,2024,60459,CT1,60.5,43.0,46.0,20.0,2017,2017,2017,<NA>
3196,2024,60459,CT2,60.5,43.0,46.0,20.0,2017,2017,2017,<NA>
3197,2024,60460,CT1,60.5,43.0,46.0,20.0,2017,2017,2017,<NA>
3198,2024,60460,CT2,60.5,43.0,46.0,20.0,2017,2017,2017,<NA>


In [11]:
# Calc min, max, and mean ages of the generators in the unit
unit_level['unit_age_min'] = unit_level["Year"] - unit_level['Operating Year_min']
unit_level['unit_age_max'] = unit_level["Year"] - unit_level['Operating Year_max']
unit_level['unit_age_mean'] = unit_level["Year"] - unit_level['Operating Year_mean']

In [12]:
unit_level = unit_level.rename(columns={
    "EIA_PLANT_ID": "Facility ID",
    "CAMD_UNIT_ID": "Unit ID"
})

In [ ]:
# Merge each with unit_level
complete_campd_ny = pd.merge(campd_ny, unit_level, on=['Year', 'Facility ID', 'Unit ID'], how='left')
complete_campd_wy = pd.merge(campd_wy, unit_level, on=['Year', 'Facility ID', 'Unit ID'], how='left')
complete_campd_tx = pd.merge(campd_tx, unit_level, on=['Year', 'Facility ID', 'Unit ID'], how='left')

0

In [ ]:
# Calculate differences between aggregated EIA Nameplate Capacity and CAMD Nameplate Capacity

# Create a column for absolute difference
complete_campd_ny["Capacity Difference"] = (
    complete_campd_ny["Nameplate Capacity (MWe)"] - complete_campd_ny["Nameplate Capacity (MW)"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_ny.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

# Calculate average difference
average_difference_ny = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference NY: {average_difference_ny:.2f} MW")

# Create a column for absolute difference
complete_campd_tx["Capacity Difference"] = (
    complete_campd_tx["Nameplate Capacity (MWe)"] - complete_campd_tx["Nameplate Capacity (MW)"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_tx.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

# Calculate average difference
average_difference_tx = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference TX: {average_difference_tx:.2f} MW")

# Create a column for absolute difference
complete_campd_wy["Capacity Difference"] = (
    complete_campd_wy["Nameplate Capacity (MWe)"] - complete_campd_wy["Nameplate Capacity (MW)"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_wy.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

# Calculate average difference
average_difference_wy = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference WY: {average_difference_wy:.2f} MW")

print("Average Difference across all: ", np.mean([average_difference_ny, average_difference_tx, average_difference_wy]))

Average Capacity Difference NY: 19.83 MW
Average Capacity Difference TX: 58.64 MW
Average Capacity Difference WY: 22.68 MW
Average Difference across all:  33.71932940077027


In [15]:
# fill EIA columns with na if it capacity difference exceeds a certain threshold

# rename the EIA column to make it clearer
for df in [complete_campd_ny, complete_campd_wy, complete_campd_tx]:
    df.rename(columns={
        "Nameplate Capacity (MW)": "Nameplate Capacity (MW) - EIA"
    }, inplace=True)

threshold = 30 
columns_to_null = [
    "Nameplate Capacity (MW) - EIA", 
    "Summer Capacity (MW)_y",
    "Winter Capacity (MW)_y",
    "Minimum Load (MW)_y"
]

# Recalculate capacity difference
complete_campd_wy["Capacity Difference"] = (
    complete_campd_wy["Nameplate Capacity (MWe)"] - complete_campd_wy["Nameplate Capacity (MW) - EIA"]
).abs()

# Identify rows with unreasonable differences
condition = complete_campd_wy["Capacity Difference"] > threshold

# Set selected columns to NaN where condition is met
complete_campd_wy.loc[condition, columns_to_null] = np.nan


# Recalculate capacity difference
complete_campd_tx["Capacity Difference"] = (
    complete_campd_tx["Nameplate Capacity (MWe)"] - complete_campd_tx["Nameplate Capacity (MW) - EIA"]
).abs()

# Identify rows with unreasonable differences
condition = complete_campd_tx["Capacity Difference"] > threshold

# Set selected columns to NaN where condition is met
complete_campd_tx.loc[condition, columns_to_null] = np.nan

# Recalculate capacity difference
complete_campd_ny["Capacity Difference"] = (
    complete_campd_ny["Nameplate Capacity (MWe)"] - complete_campd_ny["Nameplate Capacity (MW) - EIA"]
).abs()

# Identify rows with unreasonable differences
condition = complete_campd_ny["Capacity Difference"] > threshold

# Set selected columns to NaN where condition is met
complete_campd_ny.loc[condition, columns_to_null] = np.nan

In [16]:
# Calculate capacity difference again for comparison

# Create a column for absolute difference
complete_campd_ny["Capacity Difference"] = (
    complete_campd_ny["Nameplate Capacity (MWe)"] - complete_campd_ny["Nameplate Capacity (MW) - EIA"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_ny.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW) - EIA"])

# Calculate average difference
average_difference_ny = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference NY: {average_difference_ny:.2f} MW")

# Create a column for absolute difference
complete_campd_tx["Capacity Difference"] = (
    complete_campd_tx["Nameplate Capacity (MWe)"] - complete_campd_tx["Nameplate Capacity (MW) - EIA"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_tx.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW) - EIA"])

# Calculate average difference
average_difference_tx = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference TX: {average_difference_tx:.2f} MW")

# Create a column for absolute difference
complete_campd_wy["Capacity Difference"] = (
    complete_campd_wy["Nameplate Capacity (MWe)"] - complete_campd_wy["Nameplate Capacity (MW) - EIA"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_wy.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW) - EIA"])

# Calculate average difference
average_difference_wy = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference WY: {average_difference_wy:.2f} MW")


print("Average Difference across all: ", np.mean([average_difference_ny, average_difference_tx, average_difference_wy]))



Average Capacity Difference NY: 1.23 MW
Average Capacity Difference TX: 4.01 MW
Average Capacity Difference WY: 2.01 MW
Average Difference across all:  2.4185569212173195


In [ ]:
# Rename crosswalk cols and remove manual EIA transformation cols

# Define the renaming mapping
rename_cols = {
    'Summer Capacity (MW)_y': 'Summer Capacity (MW)',
    'Winter Capacity (MW)_y': 'Winter Capacity (MW)',
    'Minimum Load (MW)_y': 'Minimum Load (MW)',
    'Operating Year_min': 'Operating Year_min',
    'Operating Year_max': 'Operating Year_max',
    'Operating Year_mean': 'Operating Year_mean',
    'Planned Retirement Year_y': 'Planned Retirement Year',
    'unit_age_min_y': 'unit_age_min',
    'unit_age_max_y': 'unit_age_max',
    'unit_age_mean_y': 'unit_age_mean',
}

In [ ]:
# Clean NY dataset
complete_campd_ny = complete_campd_ny.drop(columns=[
    'Summer Capacity (MW)_x', 'Winter Capacity (MW)_x',
    'Minimum Load (MW)_x', 'Planned Retirement Year_x',
    'unit_age_min_x', 'unit_age_max_x', 'unit_age_mean_x',
    'Capacity Difference', 'Nameplate Capacity (MW) - EIA'
])
complete_campd_ny = complete_campd_ny.rename(columns=rename_cols)


In [ ]:
complete_campd_ny.to_csv('NY/Hourly/Concatted File/ny_2019_2024_complete_cleaned_NonOperating.csv', index=False)

In [20]:
# Clean WY dataset
complete_campd_wy = complete_campd_wy.drop(columns=[
    'Summer Capacity (MW)_x', 'Winter Capacity (MW)_x',
    'Minimum Load (MW)_x', 'Planned Retirement Year_x',
    'unit_age_min_x', 'unit_age_max_x', 'unit_age_mean_x',
    'Capacity Difference', 'Nameplate Capacity (MW) - EIA'
])

complete_campd_wy = complete_campd_wy.rename(columns=rename_cols)

In [21]:
complete_campd_wy.to_csv('WY/Hourly/Concatted File/wy_2019_2024_complete_cleaned_NonOperating.csv', index=False)

In [22]:
# Clean TX dataset
complete_campd_tx = complete_campd_tx.drop(columns=[
    'Summer Capacity (MW)_x', 'Winter Capacity (MW)_x',
    'Minimum Load (MW)_x', 'Planned Retirement Year_x',
    'unit_age_min_x', 'unit_age_max_x', 'unit_age_mean_x',
    'Capacity Difference', 'Nameplate Capacity (MW) - EIA'
])
complete_campd_tx = complete_campd_tx.rename(columns=rename_cols)

In [23]:
complete_campd_tx.to_csv('TX/Hourly/Concatted File/tx_2019_2024_complete_cleaned_NonOperating.csv', index=False)